# Final RuModernBERT: 3ep pretrain checkpoint -> all human labels

Финальное full fine-tuning без validation: все `365,654`
human-пар используются в train на каждой из трёх эпох. Результаты никуда,
кроме `/kaggle/working`, не отправляются.

Data: `dinakepecheva/e-cup-human-data`  
Checkpoint Dataset: `alexproger23/product-matching-rumodernbert-pretrain-3ep`

In [ ]:
import hashlib
import json
import os
import subprocess
import sys
import time
from datetime import datetime, timezone
from pathlib import Path

INPUT_ROOT = Path("/kaggle/input")
WORKING_ROOT = Path("/kaggle/working")
TEMP_ROOT = Path("/kaggle/temp/rumodernbert_3ep_full_human_final")
PROJECT_ROOT = TEMP_ROOT / "product_matching"
PREPARED_DIR = TEMP_ROOT / "prepared"
TOKEN_CACHE_DIR = TEMP_ROOT / "token_cache"
OUTPUT_DIR = WORKING_ROOT / 'rumodernbert_3ep_full_human_final'
CONFIG_PATH = WORKING_ROOT / 'rumodernbert_3ep_full_human_final_config.json'
TRAIN_LOG = WORKING_ROOT / 'rumodernbert_3ep_full_human_final.log'
EXPECTED_DATASET_REF = 'dinakepecheva/e-cup-human-data'
EXPECTED_CHECKPOINT_DATASET_REF = 'alexproger23/product-matching-rumodernbert-pretrain-3ep'
EXPECTED_CHECKPOINT_MANIFEST_SHA256 = '4fbfac78fd55130d593b5f81086edd0fffb53534c1aceeccd7ad1cfa8ccde648'
EXPECTED_DATA = {'items_rows': 711304, 'pairs_rows': 365654, 'positive_pairs': 93890, 'negative_pairs': 271764, 'duplicate_unordered_pairs': 0, 'contradictory_pairs': 0, 'items_sha256': '07d4f09ac8cfa90a2dfc2a7802dec09fc1e188576e1281e781aa1340b3336f4a', 'matches_sha256': '14b0e07f750204807aff0bb7510d385a0b3fae3f02088506a14242187ca1e492'}
EXPECTED_SOURCE_SHA256 = '122ea430328841b4c0f76369b64a5f58f5e8c4db1d0c64cac8386b224a7aa1eb'
LOCKED_RECIPE = {'epochs': 3, 'batch_size': 24, 'gradient_accumulation': 4, 'learning_rate': 4e-05, 'weight_decay': 0.01, 'warmup_ratio': 0.05, 'scheduler': 'cosine', 'max_length': 384, 'attention_implementation': 'sdpa', 'train_subset': 'all', 'sampling': 'none', 'loss_weighting': 'none', 'lexical_hard_negative_strength': 0.0, 'bucket_size_multiplier': 50, 'dataloader_workers': 4, 'prefetch_factor': 2, 'gradient_checkpointing': False, 'label_smoothing': 0.0, 'max_grad_norm': 0.5, 'seed': 42, 'skip_validation': True}
EXPECTED_AMP_DTYPE = 'torch.float16'
EXPECTED_WORLD_SIZE = 2
EXPECTED_EFFECTIVE_BATCH_SIZE = 192

def file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as source:
        for chunk in iter(lambda: source.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def verified_input(filename, expected_sha256):
    candidates = []
    for path in INPUT_ROOT.glob(f"**/{filename}"):
        if path.is_file() and file_sha256(path) == expected_sha256:
            candidates.append(path)
    unique = {str(path.resolve()): path for path in candidates}
    if len(unique) != 1:
        raise RuntimeError(
            f"Expected exactly one {filename} with SHA-256 "
            f"{expected_sha256}, found {list(unique.values())}"
        )
    return next(iter(unique.values()))

items_path = verified_input("items_human.parquet", EXPECTED_DATA["items_sha256"])
matches_path = verified_input("matches.parquet", EXPECTED_DATA["matches_sha256"])

checkpoint_manifest_candidates = [
    path
    for path in INPUT_ROOT.glob("**/checkpoint_manifest.json")
    if path.is_file()
    and file_sha256(path) == EXPECTED_CHECKPOINT_MANIFEST_SHA256
]
unique_manifests = {
    str(path.resolve()): path for path in checkpoint_manifest_candidates
}
if len(unique_manifests) != 1:
    raise RuntimeError(
        "Expected exactly one checkpoint_manifest.json for "
        f"{EXPECTED_CHECKPOINT_DATASET_REF}, found "
        f"{list(unique_manifests.values())}"
    )
checkpoint_manifest_path = next(iter(unique_manifests.values()))
checkpoint_manifest = json.loads(
    checkpoint_manifest_path.read_text(encoding="utf-8")
)
if checkpoint_manifest.get("dataset") != EXPECTED_CHECKPOINT_DATASET_REF:
    raise RuntimeError("Unexpected checkpoint Dataset reference")
checkpoint_root = checkpoint_manifest_path.parent
for filename, declaration in checkpoint_manifest["files"].items():
    checkpoint_file = checkpoint_root / filename
    if not checkpoint_file.is_file():
        raise RuntimeError(f"Checkpoint file is missing: {filename}")
    if (
        checkpoint_file.stat().st_size != declaration["bytes"]
        or file_sha256(checkpoint_file) != declaration["sha256"]
    ):
        raise RuntimeError(f"Checkpoint file differs from manifest: {filename}")
reconstruction = checkpoint_manifest.get("reconstruction")
if reconstruction:
    checkpoint_path = TEMP_ROOT / "initial_checkpoint"
    checkpoint_path.mkdir(parents=True, exist_ok=True)
    part_names = set(reconstruction["parts"])
    for filename in checkpoint_manifest["files"]:
        if filename in part_names:
            continue
        destination = checkpoint_path / filename
        if destination.exists() or destination.is_symlink():
            destination.unlink()
        destination.symlink_to(checkpoint_root / filename)
    reconstructed_model = checkpoint_path / reconstruction["filename"]
    model_digest = hashlib.sha256()
    with reconstructed_model.open("wb") as destination:
        for part_name in reconstruction["parts"]:
            with (checkpoint_root / part_name).open("rb") as source:
                for chunk in iter(lambda: source.read(8 * 1024 * 1024), b""):
                    destination.write(chunk)
                    model_digest.update(chunk)
    if (
        reconstructed_model.stat().st_size != reconstruction["bytes"]
        or model_digest.hexdigest() != reconstruction["sha256"]
    ):
        raise RuntimeError("Reconstructed checkpoint differs from manifest")
else:
    checkpoint_path = checkpoint_root
print(json.dumps({
    "items_path": str(items_path),
    "matches_path": str(matches_path),
    "checkpoint_dataset": EXPECTED_CHECKPOINT_DATASET_REF,
    "checkpoint_path": str(checkpoint_path),
    "expected_data": EXPECTED_DATA,
}, ensure_ascii=False, indent=2))
print(subprocess.run(["nvidia-smi"], check=False, capture_output=True, text=True).stdout)

## Locked final recipe

In [ ]:
TRAIN_CONFIG = {'model': 'KAGGLE_CHECKPOINT_DATASET_RESOLVED_IN_NOTEBOOK',
 'epochs': 3,
 'batch_size': 24,
 'gradient_accumulation': 4,
 'learning_rate': 4e-05,
 'weight_decay': 0.01,
 'warmup_ratio': 0.05,
 'scheduler': 'cosine',
 'max_length': 384,
 'attention_implementation': 'sdpa',
 'train_subset': 'all',
 'sampling': 'none',
 'loss_weighting': 'none',
 'lexical_hard_negative_strength': 0.0,
 'bucket_size_multiplier': 50,
 'dataloader_workers': 4,
 'prefetch_factor': 2,
 'tokenization_batch_size': 512,
 'tokenization_log_every': 50,
 'gradient_checkpointing': False,
 'label_smoothing': 0.0,
 'max_grad_norm': 0.5,
 'log_every': 20,
 'seed': 42,
 'skip_validation': True,
 'trust_remote_code': False}
TRAIN_CONFIG["model"] = str(checkpoint_path)
CONFIG_PATH.write_text(
    json.dumps(TRAIN_CONFIG, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
print(json.dumps(TRAIN_CONFIG, ensure_ascii=False, indent=2))

## Embedded trainer and dependencies

In [ ]:
EMBEDDED_SOURCES = {'requirements-cross-encoder.txt': 'transformers>=4.51.0,<5\ntorch>=2.5,<3\npandas>=2.2,<3\npyarrow>=18,<24\nscikit-learn>=1.6,<2\nnumpy>=1.26,<3\n', 'scripts/train_cross_encoder.py': '"""Configurable DDP fine-tuning for compact product-pair cross-encoders."""\n\nfrom __future__ import annotations\n\nimport argparse\nimport json\nimport math\nimport os\nimport random\nimport sys\nimport time\nfrom contextlib import nullcontext\nfrom dataclasses import dataclass\nfrom datetime import timedelta\nfrom pathlib import Path\nfrom typing import Any\n\nimport numpy as np\nimport pandas as pd\nimport torch\nimport torch.distributed as dist\nfrom sklearn.metrics import average_precision_score\nfrom torch.nn.parallel import DistributedDataParallel\nfrom torch.optim import AdamW\nfrom torch.utils.data import DataLoader\nfrom transformers import (\n    AutoConfig,\n    AutoModel,\n    AutoModelForSequenceClassification,\n    AutoTokenizer,\n    get_cosine_schedule_with_warmup,\n)\n\nsys.path.insert(0, str(Path(__file__).resolve().parents[1]))\n\nfrom src.cross_encoder_training import (\n    CrossEncoderBatchCollator,\n    CrossEncoderPairDataset,\n    PairTokenCache,\n    build_pair_token_cache,\n)\nfrom src.cross_encoder_experiment_hooks import load_loss_hook\nfrom src.data_pipeline import attach_item_fields\nfrom src.experiment_protocol import validation_split_paths\nfrom src.pair_features import (\n    build_training_loss_weights,\n    category_label_downsample,\n    name_ngram_cosine,\n)\nfrom src.qwen_reranker import preferred_cuda_dtype\nfrom src.qwen_training import (\n    FixedLengthBatchSampler,\n    LengthBucketBatchSampler,\n    balanced_sampling_weights,\n)\nfrom src.validation_metrics import binary_probability_metrics\n\n\nROOT = Path(__file__).resolve().parents[1]\nDEFAULT_CONFIG = ROOT / "configs" / "cross_encoder_minilm.json"\nCONFIG_KEYS = {\n    "model",\n    "model_backend",\n    "model_load_kwargs",\n    "trust_remote_code",\n    "classifier_dropout",\n    "epochs",\n    "batch_size",\n    "eval_batch_size",\n    "gradient_accumulation",\n    "learning_rate",\n    "weight_decay",\n    "warmup_ratio",\n    "scheduler",\n    "max_length",\n    "attention_implementation",\n    "sampling",\n    "train_subset",\n    "loss_weighting",\n    "lexical_hard_negative_strength",\n    "bucket_size_multiplier",\n    "dataloader_workers",\n    "prefetch_factor",\n    "tokenization_batch_size",\n    "tokenization_log_every",\n    "gradient_checkpointing",\n    "symmetric_validation",\n    "label_smoothing",\n    "max_grad_norm",\n    "log_every",\n    "seed",\n    "skip_validation",\n}\n\n\n@dataclass(frozen=True)\nclass ValidationResult:\n    macro_average_precision: float\n    overall_average_precision: float\n    recall_at_precision_0_99: float\n    threshold_at_precision_0_99: float | None\n    roc_auc: float\n    log_loss: float\n    per_category_average_precision: dict[str, float]\n    scores: np.ndarray\n    scores_ab: np.ndarray\n    scores_ba: np.ndarray\n    logits: np.ndarray\n    logits_ab: np.ndarray\n    logits_ba: np.ndarray\n\n\ndef load_config_from_cli() -> tuple[Path, dict[str, Any]]:\n    pre_parser = argparse.ArgumentParser(add_help=False)\n    pre_parser.add_argument("--config", type=Path, default=DEFAULT_CONFIG)\n    known, _ = pre_parser.parse_known_args()\n    config_path = known.config\n    if not config_path.is_file():\n        raise SystemExit(f"Config does not exist: {config_path}")\n    config = json.loads(config_path.read_text(encoding="utf-8"))\n    if not isinstance(config, dict):\n        raise SystemExit("Training config must contain a JSON object")\n    if unknown := set(config) - CONFIG_KEYS:\n        raise SystemExit(f"Unknown training config keys: {sorted(unknown)}")\n    return config_path, config\n\n\ndef parse_args() -> argparse.Namespace:\n    config_path, config = load_config_from_cli()\n\n    def configured(name: str, fallback: Any) -> Any:\n        return config.get(name, fallback)\n\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--config", type=Path, default=config_path)\n    parser.add_argument("--model", default=configured("model", "cross-encoder/mmarco-mMiniLMv2-L12-H384-v1"))\n    parser.add_argument(\n        "--model-backend",\n        choices=["sequence_classification", "jina_lbnl"],\n        default=configured("model_backend", "sequence_classification"),\n        help=(\n            "Model output contract. jina_lbnl uses Jina v3/v3.5\'s custom "\n            "single-document listwise prompt and cosine relevance score."\n        ),\n    )\n    parser.add_argument(\n        "--trust-remote-code",\n        action=argparse.BooleanOptionalAction,\n        default=configured("trust_remote_code", False),\n        help="Allow Hugging Face model/tokenizer repositories to execute custom code",\n    )\n    parser.add_argument(\n        "--classifier-dropout",\n        type=float,\n        default=configured("classifier_dropout", None),\n        help="Explicit classifier-head dropout; omit to keep the checkpoint config",\n    )\n    parser.add_argument(\n        "--model-load-kwarg",\n        action="append",\n        default=[],\n        metavar="NAME=JSON_VALUE",\n        help=(\n            "Extra model-specific from_pretrained keyword; repeat as needed. "\n            "For example: use_flash_attn=false"\n        ),\n    )\n    parser.add_argument("--prepared-dir", type=Path, default=Path("prepared/human"))\n    parser.add_argument("--output-dir", type=Path, default=Path("models/cross_encoder_minilm"))\n    parser.add_argument("--token-cache-dir", type=Path)\n    parser.add_argument(\n        "--loss-hook",\n        type=Path,\n        help=(\n            "Optional Python module defining compute_loss(...), and optionally "\n            "initialize_loss(...), for controlled loss ablations"\n        ),\n    )\n    parser.add_argument(\n        "--validation-split",\n        action="append",\n        default=[],\n        metavar="NAME=PATH",\n        help=(\n            "Named validation pair file; repeat for IID, hard and OOD. Relative "\n            "paths are resolved below --prepared-dir. Defaults to iid=val_pairs.parquet."\n        ),\n    )\n    parser.add_argument(\n        "--skip-validation",\n        action=argparse.BooleanOptionalAction,\n        default=configured("skip_validation", False),\n        help="Train and save the model without loading or evaluating validation data",\n    )\n    parser.add_argument("--epochs", type=int, default=configured("epochs", 1))\n    parser.add_argument("--batch-size", type=int, default=configured("batch_size", 96), help="Per GPU")\n    parser.add_argument("--eval-batch-size", type=int, default=configured("eval_batch_size", 192), help="Per GPU")\n    parser.add_argument(\n        "--gradient-accumulation",\n        type=int,\n        default=configured("gradient_accumulation", 1),\n    )\n    parser.add_argument("--learning-rate", type=float, default=configured("learning_rate", 2e-5))\n    parser.add_argument("--weight-decay", type=float, default=configured("weight_decay", 0.01))\n    parser.add_argument("--warmup-ratio", type=float, default=configured("warmup_ratio", 0.05))\n    parser.add_argument(\n        "--scheduler",\n        choices=["cosine"],\n        default=configured("scheduler", "cosine"),\n    )\n    parser.add_argument("--max-length", type=int, default=configured("max_length", 256))\n    parser.add_argument(\n        "--attention-implementation",\n        choices=["eager", "sdpa"],\n        default=configured("attention_implementation", "sdpa"),\n    )\n    parser.add_argument(\n        "--sampling",\n        choices=["none", "category", "category_label"],\n        default=configured("sampling", "category_label"),\n    )\n    parser.add_argument(\n        "--train-subset",\n        choices=["all", "category_label_downsample"],\n        default=configured("train_subset", "all"),\n        help="Optional deterministic filtering before tokenization",\n    )\n    parser.add_argument(\n        "--loss-weighting",\n        choices=["none", "category_label_sqrt"],\n        default=configured("loss_weighting", "none"),\n        help="Per-example BCE weighting; unlike sampling, retains every row",\n    )\n    parser.add_argument(\n        "--lexical-hard-negative-strength",\n        type=float,\n        default=configured("lexical_hard_negative_strength", 0.0),\n        help="Within-category emphasis for lexically similar negative pairs",\n    )\n    parser.add_argument(\n        "--bucket-size-multiplier",\n        type=int,\n        default=configured("bucket_size_multiplier", 50),\n    )\n    parser.add_argument(\n        "--dataloader-workers",\n        type=int,\n        default=configured("dataloader_workers", 2),\n        help="Per DDP process",\n    )\n    parser.add_argument("--prefetch-factor", type=int, default=configured("prefetch_factor", 2))\n    parser.add_argument(\n        "--tokenization-batch-size",\n        type=int,\n        default=configured("tokenization_batch_size", 512),\n    )\n    parser.add_argument(\n        "--tokenization-log-every",\n        type=int,\n        default=configured("tokenization_log_every", 50),\n    )\n    parser.add_argument(\n        "--gradient-checkpointing",\n        action=argparse.BooleanOptionalAction,\n        default=configured("gradient_checkpointing", False),\n    )\n    parser.add_argument(\n        "--symmetric-validation",\n        action=argparse.BooleanOptionalAction,\n        default=configured("symmetric_validation", True),\n    )\n    parser.add_argument(\n        "--label-smoothing", type=float, default=configured("label_smoothing", 0.0)\n    )\n    parser.add_argument("--max-grad-norm", type=float, default=configured("max_grad_norm", 1.0))\n    parser.add_argument("--log-every", type=int, default=configured("log_every", 20))\n    parser.add_argument("--seed", type=int, default=configured("seed", 42))\n    return parser.parse_args()\n\n\ndef validate_args(args: argparse.Namespace) -> None:\n    positive = {\n        "epochs": args.epochs,\n        "batch-size": args.batch_size,\n        "eval-batch-size": args.eval_batch_size,\n        "gradient-accumulation": args.gradient_accumulation,\n        "learning-rate": args.learning_rate,\n        "max-length": args.max_length,\n        "bucket-size-multiplier": args.bucket_size_multiplier,\n        "prefetch-factor": args.prefetch_factor,\n        "tokenization-batch-size": args.tokenization_batch_size,\n        "tokenization-log-every": args.tokenization_log_every,\n        "max-grad-norm": args.max_grad_norm,\n        "log-every": args.log_every,\n    }\n    if invalid := [name for name, value in positive.items() if value <= 0]:\n        raise ValueError(f"These arguments must be positive: {\', \'.join(invalid)}")\n    if args.dataloader_workers < 0:\n        raise ValueError("dataloader-workers must be non-negative")\n    if args.weight_decay < 0:\n        raise ValueError("weight-decay must be non-negative")\n    if args.lexical_hard_negative_strength < 0:\n        raise ValueError("lexical-hard-negative-strength must be non-negative")\n    if not 0 <= args.warmup_ratio < 1:\n        raise ValueError("warmup-ratio must be in [0, 1)")\n    if not 0 <= args.label_smoothing < 1:\n        raise ValueError("label-smoothing must be in [0, 1)")\n    if args.classifier_dropout is not None and not 0 <= args.classifier_dropout < 1:\n        raise ValueError("classifier-dropout must be in [0, 1)")\n\n\ndef model_load_kwargs(args: argparse.Namespace) -> dict[str, Any]:\n    configured = json.loads(args.config.read_text(encoding="utf-8")).get(\n        "model_load_kwargs", {}\n    )\n    if not isinstance(configured, dict):\n        raise ValueError("model_load_kwargs in the config must be a JSON object")\n    result = dict(configured)\n    for spec in args.model_load_kwarg:\n        name, separator, raw_value = spec.partition("=")\n        name = name.strip()\n        if not separator or not name:\n            raise ValueError(\n                f"Invalid --model-load-kwarg {spec!r}; expected NAME=JSON_VALUE"\n            )\n        try:\n            result[name] = json.loads(raw_value)\n        except json.JSONDecodeError as error:\n            raise ValueError(\n                f"Invalid JSON value in --model-load-kwarg {spec!r}"\n            ) from error\n    reserved = {\n        "pretrained_model_name_or_path",\n        "num_labels",\n        "attn_implementation",\n        "trust_remote_code",\n    }\n    if conflict := reserved & set(result):\n        raise ValueError(f"Reserved model-load kwargs cannot be overridden: {sorted(conflict)}")\n    return result\n\n\ndef loader_options(args: argparse.Namespace, *, persistent: bool) -> dict[str, Any]:\n    options: dict[str, Any] = {\n        "num_workers": args.dataloader_workers,\n        "pin_memory": True,\n    }\n    if args.dataloader_workers:\n        options.update(\n            persistent_workers=persistent,\n            prefetch_factor=args.prefetch_factor,\n        )\n    return options\n\n\ndef create_caches(\n    train: pd.DataFrame,\n    validations: dict[str, pd.DataFrame],\n    tokenizer: Any,\n    args: argparse.Namespace,\n    is_main: bool,\n    distributed: bool,\n    control_group: dist.ProcessGroup | None,\n) -> tuple[PairTokenCache, dict[str, PairTokenCache]]:\n    cache_root = args.token_cache_dir or (\n        Path("artifacts/token_cache") / args.output_dir.name\n    )\n    payload: dict[str, Any] | None = None\n    local_error: Exception | None = None\n    if is_main:\n        try:\n            train_cache = build_pair_token_cache(\n                train,\n                tokenizer,\n                cache_root,\n                "train",\n                args.model,\n                args.max_length,\n                args.tokenization_batch_size,\n                args.tokenization_log_every,\n                pair_format=(\n                    "jina_lbnl"\n                    if args.model_backend == "jina_lbnl"\n                    else "cross_encoder"\n                ),\n            )\n            validation_caches = {\n                name: build_pair_token_cache(\n                    validation,\n                    tokenizer,\n                    cache_root,\n                    f"validation_{name}",\n                    args.model,\n                    args.max_length,\n                    args.tokenization_batch_size,\n                    args.tokenization_log_every,\n                    pair_format=(\n                        "jina_lbnl"\n                        if args.model_backend == "jina_lbnl"\n                        else "cross_encoder"\n                    ),\n                )\n                for name, validation in validations.items()\n            }\n            payload = {\n                "train_path": str(train_cache.directory),\n                "validation_paths": {\n                    name: str(cache.directory)\n                    for name, cache in validation_caches.items()\n                },\n                "error": None,\n            }\n        except Exception as error:\n            local_error = error\n            payload = {\n                "paths": None,\n                "error": f"{type(error).__name__}: {error}",\n            }\n    if distributed:\n        if control_group is None:\n            raise RuntimeError("Distributed cache coordination requires a control group")\n        message: list[Any] = [payload]\n        # Tokenization can take longer than NCCL\'s normal collective timeout.\n        # Keep this CPU-only coordination off the GPU process group.\n        dist.broadcast_object_list(message, src=0, group=control_group)\n        payload = message[0]\n    if payload is None:\n        raise RuntimeError("Token cache paths were not initialized")\n    if payload["error"] is not None:\n        if local_error is not None:\n            raise local_error\n        raise RuntimeError(f"Rank 0 failed to create token caches: {payload[\'error\']}")\n    train_path = payload.get("train_path")\n    validation_paths = payload.get("validation_paths")\n    if not isinstance(train_path, str) or not isinstance(validation_paths, dict):\n        raise RuntimeError(f"Invalid token cache payload: {payload}")\n    if set(validation_paths) != set(validations):\n        raise RuntimeError(f"Validation token cache payload differs: {payload}")\n    return PairTokenCache.load(Path(train_path)), {\n        name: PairTokenCache.load(Path(validation_paths[name]))\n        for name in validations\n    }\n\n\ndef evaluate(\n    model: torch.nn.Module,\n    loader: DataLoader,\n    targets: np.ndarray,\n    categories: list[str],\n    device: torch.device,\n    amp_dtype: torch.dtype,\n    distributed: bool,\n    world_size: int,\n    model_backend: str,\n) -> ValidationResult | None:\n    model.eval()\n    local: list[tuple[int, bool, float, float]] = []\n    with torch.inference_mode():\n        for packed in loader:\n            pair_indices = packed.pop("pair_indices").tolist()\n            orientations = packed.pop("orientations").tolist()\n            packed.pop("targets")\n            packed.pop("sample_weights")\n            batch = {\n                key: value.to(device, non_blocking=True)\n                for key, value in packed.items()\n            }\n            with torch.autocast(device_type="cuda", dtype=amp_dtype):\n                logits = relevance_logits(model(**batch), model_backend)\n            raw_logits = logits.float().cpu()\n            probabilities = raw_logits.sigmoid().tolist()\n            local.extend(\n                (int(index), bool(reverse), float(probability), float(raw_logit))\n                for index, reverse, probability, raw_logit in zip(\n                    pair_indices, orientations, probabilities, raw_logits.tolist()\n                )\n            )\n\n    if distributed:\n        gathered: list[Any] = [None] * world_size\n        dist.all_gather_object(gathered, local)\n    else:\n        gathered = [local]\n    if distributed and dist.get_rank() != 0:\n        return None\n\n    scores_ab = np.full(len(targets), np.nan, dtype=np.float32)\n    scores_ba = np.full(len(targets), np.nan, dtype=np.float32)\n    logits_ab = np.full(len(targets), np.nan, dtype=np.float32)\n    logits_ba = np.full(len(targets), np.nan, dtype=np.float32)\n    for part in gathered:\n        for index, reverse, score, raw_logit in part:\n            score_destination = scores_ba if reverse else scores_ab\n            logit_destination = logits_ba if reverse else logits_ab\n            if np.isfinite(score_destination[index]):\n                raise RuntimeError(\n                    f"Validation produced a duplicate score for pair {index}, "\n                    f"reverse={reverse}"\n                )\n            score_destination[index] = score\n            logit_destination[index] = raw_logit\n    if not np.isfinite(scores_ab).all():\n        missing = int((~np.isfinite(scores_ab)).sum())\n        raise RuntimeError(f"Validation produced {missing} missing A/B scores")\n    has_reverse_scores = bool(np.isfinite(scores_ba).any())\n    if has_reverse_scores and not np.isfinite(scores_ba).all():\n        missing = int((~np.isfinite(scores_ba)).sum())\n        raise RuntimeError(f"Validation produced {missing} missing B/A scores")\n    scores = (\n        (scores_ab + scores_ba) / 2.0\n        if has_reverse_scores\n        else scores_ab.copy()\n    )\n    logits = (\n        (logits_ab + logits_ba) / 2.0\n        if has_reverse_scores\n        else logits_ab.copy()\n    )\n    frame = pd.DataFrame({"target": targets, "predict": scores, "category": categories})\n    per_category = frame.groupby("category").apply(\n        lambda group: average_precision_score(group["target"], group["predict"]),\n        include_groups=False,\n    )\n    overall_ap = float(average_precision_score(targets, scores))\n    probability_metrics = binary_probability_metrics(targets, scores)\n    return ValidationResult(\n        macro_average_precision=float(per_category.mean()),\n        overall_average_precision=overall_ap,\n        recall_at_precision_0_99=probability_metrics[\n            "recall_at_precision_0_99"\n        ],\n        threshold_at_precision_0_99=probability_metrics[\n            "threshold_at_precision_0_99"\n        ],\n        roc_auc=probability_metrics["roc_auc"],\n        log_loss=probability_metrics["log_loss"],\n        per_category_average_precision={\n            str(key): float(value) for key, value in per_category.items()\n        },\n        scores=scores,\n        scores_ab=scores_ab,\n        scores_ba=scores_ba,\n        logits=logits,\n        logits_ab=logits_ab,\n        logits_ba=logits_ba,\n    )\n\n\ndef relevance_logits(outputs: Any, model_backend: str) -> torch.Tensor:\n    if model_backend == "sequence_classification":\n        logits = getattr(outputs, "logits", None)\n    elif model_backend == "jina_lbnl":\n        logits = getattr(outputs, "scores", None)\n    else:\n        raise ValueError(f"Unknown model backend: {model_backend!r}")\n    if logits is None:\n        raise RuntimeError(\n            f"Model backend {model_backend!r} did not return its expected score tensor"\n        )\n    if logits.ndim == 2 and logits.shape[-1] == 1:\n        logits = logits[:, 0]\n    if logits.ndim != 1:\n        raise RuntimeError(\n            f"Model backend {model_backend!r} returned scores with shape "\n            f"{tuple(logits.shape)}; expected one score per pair"\n        )\n    return logits\n\n\ndef build_validation_predictions(\n    validation: pd.DataFrame,\n    validation_cache: PairTokenCache,\n    result: ValidationResult,\n    max_length: int,\n) -> pd.DataFrame:\n    """Combine validation metadata and model scores into an analysis-ready table."""\n    columns = [\n        "id1",\n        "id2",\n        "target",\n        "category_1",\n        "category_2",\n        "product_text_1",\n        "product_text_2",\n    ]\n    missing = [column for column in columns if column not in validation]\n    if missing:\n        raise ValueError(f"Validation metadata is missing columns: {missing}")\n    if len(validation) != len(result.scores):\n        raise ValueError("Validation metadata and prediction lengths differ")\n\n    predictions = validation[columns].reset_index(drop=True).copy()\n    predictions.insert(0, "pair_index", np.arange(len(predictions), dtype=np.int64))\n    predictions["score"] = result.scores\n    predictions["score_ab"] = result.scores_ab\n    predictions["score_ba"] = result.scores_ba\n    predictions["logit"] = result.logits\n    predictions["logit_ab"] = result.logits_ab\n    predictions["logit_ba"] = result.logits_ba\n    predictions["probability"] = result.scores\n    predictions["probability_ab"] = result.scores_ab\n    predictions["probability_ba"] = result.scores_ba\n    predictions["score_order_gap"] = np.abs(result.scores_ab - result.scores_ba)\n    predictions["token_length_ab"] = validation_cache.forward_lengths.astype(np.int32)\n    predictions["token_length_ba"] = validation_cache.reverse_lengths.astype(np.int32)\n    predictions["reached_max_length_ab"] = (\n        predictions["token_length_ab"] >= max_length\n    )\n    predictions["reached_max_length_ba"] = (\n        predictions["token_length_ba"] >= max_length\n    )\n    return predictions\n\n\ndef main() -> None:\n    args = parse_args()\n    validate_args(args)\n    pipeline_started = time.perf_counter()\n    if not torch.cuda.is_available():\n        raise RuntimeError("CUDA GPU is required")\n\n    distributed = int(os.environ.get("WORLD_SIZE", "1")) > 1\n    local_rank = int(os.environ.get("LOCAL_RANK", "0"))\n    control_group: dist.ProcessGroup | None = None\n    if distributed:\n        torch.cuda.set_device(local_rank)\n        process_group_timeout = timedelta(hours=1)\n        dist.init_process_group(backend="nccl", timeout=process_group_timeout)\n        control_group = dist.new_group(\n            backend="gloo",\n            timeout=process_group_timeout,\n        )\n    rank = dist.get_rank() if distributed else 0\n    world_size = dist.get_world_size() if distributed else 1\n    device = torch.device("cuda", local_rank)\n    is_main = rank == 0\n    random.seed(args.seed + rank)\n    np.random.seed(args.seed + rank)\n    torch.manual_seed(args.seed + rank)\n\n    items = pd.read_parquet(\n        args.prepared_dir / "items.parquet",\n        columns=["id", "product_text", "category"],\n    )\n    train_pairs = pd.read_parquet(args.prepared_dir / "train_pairs.parquet")\n    validation_paths = (\n        {}\n        if args.skip_validation\n        else validation_split_paths(args.prepared_dir, args.validation_split)\n    )\n    missing_validation_files = [\n        str(path) for path in validation_paths.values() if not path.is_file()\n    ]\n    if missing_validation_files:\n        raise FileNotFoundError(\n            f"Validation pair files do not exist: {missing_validation_files}"\n        )\n    train = attach_item_fields(\n        train_pairs, items, fields=("product_text", "category")\n    )\n    validations = {\n        name: attach_item_fields(\n            pd.read_parquet(path), items, fields=("product_text", "category")\n        )\n        for name, path in validation_paths.items()\n    }\n    if (train["category_1"] != train["category_2"]).any():\n        raise ValueError("Training contains cross-category pairs")\n    if not train["target"].between(0, 1).all():\n        raise ValueError("Training targets must be probabilities in [0, 1]")\n    for name, validation in validations.items():\n        if (validation["category_1"] != validation["category_2"]).any():\n            raise ValueError(f"Validation split {name!r} contains cross-category pairs")\n        if not validation["target"].isin([0.0, 1.0]).all():\n            raise ValueError(\n                f"Validation split {name!r} targets must be binary for average precision"\n            )\n    original_train_pairs = len(train)\n    if args.train_subset == "category_label_downsample":\n        train = category_label_downsample(train, seed=args.seed)\n    train = train.reset_index(drop=True)\n\n    tokenizer = AutoTokenizer.from_pretrained(\n        args.model,\n        use_fast=True,\n        trust_remote_code=args.trust_remote_code,\n    )\n    if tokenizer.pad_token_id is None:\n        if args.model_backend == "jina_lbnl" and tokenizer.unk_token_id is not None:\n            tokenizer.pad_token = tokenizer.unk_token\n        else:\n            raise ValueError("Cross-encoder tokenizer must define pad_token_id")\n    tokenization_started = time.perf_counter()\n    train_cache, validation_caches = create_caches(\n        train,\n        validations,\n        tokenizer,\n        args,\n        is_main,\n        distributed,\n        control_group,\n    )\n    tokenization_seconds = time.perf_counter() - tokenization_started\n    os.environ["TOKENIZERS_PARALLELISM"] = "false"\n\n    train_targets = train["target"].to_numpy(dtype=np.float32)\n    train_categories = train["category_1"].astype(str).tolist()\n    validation_targets = {\n        name: validation["target"].to_numpy(dtype=np.float32)\n        for name, validation in validations.items()\n    }\n    validation_categories = {\n        name: validation["category_1"].astype(str).tolist()\n        for name, validation in validations.items()\n    }\n    lexical_similarities = (\n        name_ngram_cosine(train)\n        if args.lexical_hard_negative_strength > 0\n        else None\n    )\n    training_loss_weights = build_training_loss_weights(\n        train_categories,\n        train_targets,\n        mode=args.loss_weighting,\n        lexical_similarities=lexical_similarities,\n        lexical_hard_negative_strength=args.lexical_hard_negative_strength,\n    )\n    data_sample_weights = (\n        train["sample_weight"].to_numpy(dtype=np.float32)\n        if "sample_weight" in train\n        else np.ones(len(train), dtype=np.float32)\n    )\n    if not np.isfinite(data_sample_weights).all() or (data_sample_weights <= 0).any():\n        raise ValueError("Training sample_weight values must be finite and positive")\n    training_loss_weights *= data_sample_weights\n    training_loss_weights /= training_loss_weights.mean()\n    training_source_counts = (\n        {str(key): int(value) for key, value in train["label_source"].value_counts().items()}\n        if "label_source" in train\n        else {"unspecified": len(train)}\n    )\n    training_source_weight_mass = (\n        {\n            str(source): float(training_loss_weights[positions].sum())\n            for source, positions in train.groupby("label_source").indices.items()\n        }\n        if "label_source" in train\n        else {"unspecified": float(training_loss_weights.sum())}\n    )\n    sampling_weights = balanced_sampling_weights(\n        train_categories, train_targets, args.sampling\n    )\n    train_dataset = CrossEncoderPairDataset(\n        train_cache,\n        train_targets,\n        sample_weights=training_loss_weights,\n    )\n    validation_datasets = {\n        name: CrossEncoderPairDataset(\n            validation_caches[name], validation_targets[name]\n        )\n        for name in validations\n    }\n    train_sampler = LengthBucketBatchSampler(\n        train_cache.forward_lengths,\n        train_cache.reverse_lengths,\n        batch_size=args.batch_size,\n        rank=rank,\n        world_size=world_size,\n        weights=sampling_weights,\n        bucket_size_multiplier=args.bucket_size_multiplier,\n        seed=args.seed,\n    )\n    validation_samplers = {\n        name: FixedLengthBatchSampler(\n            validation_caches[name],\n            np.arange(rank, len(validation_datasets[name]), world_size),\n            args.eval_batch_size,\n            both_orientations=args.symmetric_validation,\n        )\n        for name in validations\n    }\n    collator = CrossEncoderBatchCollator(tokenizer.pad_token_id)\n    train_loader = DataLoader(\n        train_dataset,\n        batch_sampler=train_sampler,\n        collate_fn=collator,\n        **loader_options(args, persistent=True),\n    )\n    validation_loaders = {\n        name: DataLoader(\n            validation_datasets[name],\n            batch_sampler=validation_samplers[name],\n            collate_fn=collator,\n            **loader_options(args, persistent=False),\n        )\n        for name in validations\n    }\n    loss_hook = load_loss_hook(args.loss_hook)\n    loss_hook.initialize(\n        train_frame=train,\n        device=device,\n        rank=rank,\n        world_size=world_size,\n    )\n    del train, items\n\n    amp_dtype = preferred_cuda_dtype()\n    extra_model_load_kwargs = model_load_kwargs(args)\n    if args.model_backend == "sequence_classification":\n        model_config = AutoConfig.from_pretrained(\n            args.model,\n            trust_remote_code=args.trust_remote_code,\n        )\n        model_config.num_labels = 1\n        if args.classifier_dropout is not None:\n            model_config.classifier_dropout = args.classifier_dropout\n        model = AutoModelForSequenceClassification.from_pretrained(\n            args.model,\n            config=model_config,\n            attn_implementation=args.attention_implementation,\n            trust_remote_code=args.trust_remote_code,\n            **extra_model_load_kwargs,\n        )\n        model.config.id2label = {0: "MATCH_SCORE"}\n        model.config.label2id = {"MATCH_SCORE": 0}\n    else:\n        if not args.trust_remote_code:\n            raise ValueError("jina_lbnl backend requires --trust-remote-code")\n        model = AutoModel.from_pretrained(\n            args.model,\n            attn_implementation=args.attention_implementation,\n            trust_remote_code=True,\n            **extra_model_load_kwargs,\n        )\n        # Jina\'s remote forward replaces the unused causal LM head with Identity.\n        # Do it before optimizer construction so those parameters never consume\n        # optimizer state during pairwise fine-tuning.\n        if hasattr(model, "lm_head"):\n            model.lm_head = torch.nn.Identity()\n    model = model.to(device)\n    if args.gradient_checkpointing:\n        if hasattr(model.config, "use_cache"):\n            model.config.use_cache = False\n        model.gradient_checkpointing_enable(\n            gradient_checkpointing_kwargs={"use_reentrant": False}\n        )\n\n    training_model: torch.nn.Module = model\n    if distributed:\n        training_model = DistributedDataParallel(\n            training_model,\n            device_ids=[local_rank],\n            output_device=local_rank,\n            broadcast_buffers=False,\n            gradient_as_bucket_view=True,\n        )\n    named_parameters = [\n        (name, parameter)\n        for name, parameter in training_model.named_parameters()\n        if parameter.requires_grad\n    ]\n    decay_parameters = [\n        parameter\n        for name, parameter in named_parameters\n        if parameter.ndim >= 2 and "layer_norm" not in name.lower()\n    ]\n    no_decay_parameters = [\n        parameter\n        for name, parameter in named_parameters\n        if parameter.ndim < 2 or "layer_norm" in name.lower()\n    ]\n    optimizer = AdamW(\n        [\n            {"params": decay_parameters, "weight_decay": args.weight_decay},\n            {"params": no_decay_parameters, "weight_decay": 0.0},\n        ],\n        lr=args.learning_rate,\n    )\n    updates_per_epoch = math.ceil(len(train_loader) / args.gradient_accumulation)\n    total_updates = updates_per_epoch * args.epochs\n    scheduler = get_cosine_schedule_with_warmup(\n        optimizer,\n        max(1, int(total_updates * args.warmup_ratio)),\n        total_updates,\n    )\n    scaler = torch.amp.GradScaler("cuda", enabled=amp_dtype == torch.float16)\n    trainable_parameters = [parameter for _, parameter in named_parameters]\n\n    if is_main:\n        print(\n            json.dumps(\n                {\n                    "gpu": torch.cuda.get_device_name(device),\n                    "world_size": world_size,\n                    "model": args.model,\n                    "model_backend": args.model_backend,\n                    "architecture": type(model).__name__,\n                    "classifier_dropout": getattr(\n                        model.config, "classifier_dropout", None\n                    ),\n                    "amp_dtype": str(amp_dtype),\n                    "trainable_parameters": sum(p.numel() for p in trainable_parameters),\n                    "train_pairs": len(train_dataset),\n                    "original_train_pairs": original_train_pairs,\n                    "train_subset": args.train_subset,\n                    "validation_pairs": {\n                        name: len(dataset)\n                        for name, dataset in validation_datasets.items()\n                    },\n                    "steps_per_epoch": len(train_loader),\n                    "per_device_batch": args.batch_size,\n                    "effective_batch": args.batch_size\n                    * world_size\n                    * args.gradient_accumulation,\n                    "dataloader_workers_total": args.dataloader_workers * world_size,\n                    "validation_schedule": (\n                        "skipped" if args.skip_validation else "after_training"\n                    ),\n                    "sampling": args.sampling,\n                    "loss_weighting": args.loss_weighting,\n                    "loss_hook": loss_hook.metadata,\n                    "sampler_unique_coverage_per_epoch": (\n                        1.0 if args.sampling == "none" else None\n                    ),\n                    "loss_weight_min": float(training_loss_weights.min()),\n                    "loss_weight_median": float(np.median(training_loss_weights)),\n                    "loss_weight_max": float(training_loss_weights.max()),\n                    "training_source_counts": training_source_counts,\n                    "training_source_weight_mass": training_source_weight_mass,\n                    "weighted_positive_fraction": float(\n                        training_loss_weights[train_targets >= 0.5].sum()\n                        / training_loss_weights.sum()\n                    ),\n                    "negative_name_ngram_cosine_quantiles": (\n                        {\n                            str(quantile): float(value)\n                            for quantile, value in zip(\n                                (0.5, 0.9, 0.99),\n                                np.quantile(\n                                    lexical_similarities[train_targets < 0.5],\n                                    (0.5, 0.9, 0.99),\n                                ),\n                            )\n                        }\n                        if lexical_similarities is not None\n                        else None\n                    ),\n                }\n            ),\n            flush=True,\n        )\n\n    torch.cuda.reset_peak_memory_stats(device)\n    torch.cuda.synchronize(device)\n    training_started = time.perf_counter()\n    local_examples = local_useful_tokens = local_padded_tokens = 0\n    optimizer.zero_grad(set_to_none=True)\n\n    for epoch in range(args.epochs):\n        train_sampler.set_epoch(epoch)\n        training_model.train()\n        interval_started = time.perf_counter()\n        interval_loss = torch.zeros((), dtype=torch.float32, device=device)\n        interval_loss_metrics: dict[str, torch.Tensor] = {}\n        interval_steps = interval_examples = 0\n        interval_useful_tokens = interval_padded_tokens = 0\n        previous_step_finished = interval_started\n        interval_data_seconds = 0.0\n\n        for step, packed in enumerate(train_loader):\n            batch_ready = time.perf_counter()\n            interval_data_seconds += batch_ready - previous_step_finished\n            pair_indices = packed.pop("pair_indices")\n            orientations = packed.pop("orientations")\n            if loss_hook.path is not None:\n                pair_indices = pair_indices.to(device, non_blocking=True)\n                orientations = orientations.to(device, non_blocking=True)\n            cpu_targets = packed.pop("targets")\n            batch_examples = len(cpu_targets)\n            useful_tokens = int(packed["attention_mask"].sum())\n            padded_tokens = packed["attention_mask"].numel()\n            targets = cpu_targets.to(device, non_blocking=True)\n            if args.label_smoothing:\n                targets = targets * (1 - args.label_smoothing) + 0.5 * args.label_smoothing\n            weights = packed.pop("sample_weights").to(device, non_blocking=True)\n            batch = {\n                key: value.to(device, non_blocking=True)\n                for key, value in packed.items()\n            }\n            group_start = (step // args.gradient_accumulation) * args.gradient_accumulation\n            group_size = min(\n                args.gradient_accumulation, len(train_loader) - group_start\n            )\n            should_update = (\n                (step + 1) % args.gradient_accumulation == 0\n                or step + 1 == len(train_loader)\n            )\n            sync_context = (\n                training_model.no_sync()\n                if distributed and not should_update\n                else nullcontext()\n            )\n            with sync_context:\n                with torch.autocast(device_type="cuda", dtype=amp_dtype):\n                    logits = relevance_logits(\n                        training_model(**batch), args.model_backend\n                    )\n                    raw_loss, loss_metrics = loss_hook.compute(\n                        logits=logits.float(),\n                        targets=targets,\n                        sample_weights=weights,\n                        pair_indices=pair_indices,\n                        orientations=orientations,\n                        epoch=epoch,\n                        step=step,\n                    )\n                    loss = raw_loss / group_size\n                scaler.scale(loss).backward()\n\n            if should_update:\n                scaler.unscale_(optimizer)\n                torch.nn.utils.clip_grad_norm_(\n                    trainable_parameters, args.max_grad_norm\n                )\n                scale_before_step = scaler.get_scale()\n                scaler.step(optimizer)\n                scaler.update()\n                # FP16 GradScaler may skip its first overflowing optimizer step.\n                # Advance the LR schedule only when optimizer.step actually ran.\n                if scaler.get_scale() >= scale_before_step:\n                    scheduler.step()\n                optimizer.zero_grad(set_to_none=True)\n\n            local_examples += batch_examples\n            local_useful_tokens += useful_tokens\n            local_padded_tokens += padded_tokens\n            interval_examples += batch_examples\n            interval_useful_tokens += useful_tokens\n            interval_padded_tokens += padded_tokens\n            interval_loss += raw_loss.detach()\n            for name, value in loss_metrics.items():\n                interval_loss_metrics[name] = (\n                    interval_loss_metrics.get(name, torch.zeros_like(value)) + value\n                )\n            interval_steps += 1\n            previous_step_finished = time.perf_counter()\n\n            if is_main and (\n                (step + 1) % args.log_every == 0 or step + 1 == len(train_loader)\n            ):\n                torch.cuda.synchronize(device)\n                interval_seconds = time.perf_counter() - interval_started\n                seconds_per_step = interval_seconds / interval_steps\n                log_record = {\n                    "epoch": epoch + 1,\n                    "step": step + 1,\n                    "steps": len(train_loader),\n                    "loss": float(interval_loss) / interval_steps,\n                    "examples_per_second": interval_examples\n                    * world_size\n                    / interval_seconds,\n                    "seconds_per_step": seconds_per_step,\n                    "epoch_eta_minutes": (len(train_loader) - step - 1)\n                    * seconds_per_step\n                    / 60,\n                    "data_wait_fraction_approx": interval_data_seconds\n                    / interval_seconds,\n                    "padding_efficiency": interval_useful_tokens\n                    / interval_padded_tokens,\n                    "peak_vram_gib": torch.cuda.max_memory_allocated(device)\n                    / 2**30,\n                    "learning_rate": scheduler.get_last_lr()[0],\n                }\n                if interval_loss_metrics:\n                    log_record["loss_metrics"] = {\n                        name: float(value) / interval_steps\n                        for name, value in sorted(interval_loss_metrics.items())\n                    }\n                print(json.dumps(log_record), flush=True)\n                interval_started = time.perf_counter()\n                interval_loss = torch.zeros((), dtype=torch.float32, device=device)\n                interval_loss_metrics = {}\n                interval_steps = interval_examples = 0\n                interval_useful_tokens = interval_padded_tokens = 0\n                interval_data_seconds = 0.0\n                previous_step_finished = interval_started\n\n    torch.cuda.synchronize(device)\n    training_seconds = time.perf_counter() - training_started\n    totals = torch.tensor(\n        [local_examples, local_useful_tokens, local_padded_tokens],\n        dtype=torch.float64,\n        device=device,\n    )\n    elapsed = torch.tensor(training_seconds, dtype=torch.float64, device=device)\n    if distributed:\n        dist.all_reduce(totals, op=dist.ReduceOp.SUM)\n        dist.all_reduce(elapsed, op=dist.ReduceOp.MAX)\n\n    del train_loader\n    inference_model = training_model.module if distributed else training_model\n    torch.cuda.synchronize(device)\n    validation_started = time.perf_counter()\n    validation_results: dict[str, ValidationResult] = {}\n    validation_seconds_by_split: dict[str, float] = {}\n    for name, validation_loader in validation_loaders.items():\n        split_started = time.perf_counter()\n        validation_result = evaluate(\n            inference_model,\n            validation_loader,\n            validation_targets[name],\n            validation_categories[name],\n            device,\n            amp_dtype,\n            distributed,\n            world_size,\n            args.model_backend,\n        )\n        torch.cuda.synchronize(device)\n        validation_seconds_by_split[name] = time.perf_counter() - split_started\n        if is_main:\n            if validation_result is None:\n                raise RuntimeError(\n                    f"Main rank did not receive metrics for validation split {name!r}"\n                )\n            validation_results[name] = validation_result\n    if distributed:\n        dist.barrier()\n    torch.cuda.synchronize(device)\n    validation_seconds = time.perf_counter() - validation_started\n    peak_memory = torch.tensor(\n        [torch.cuda.max_memory_allocated(device) / 2**30],\n        dtype=torch.float64,\n        device=device,\n    )\n    if distributed:\n        gathered_peak_memory = [\n            torch.zeros_like(peak_memory) for _ in range(world_size)\n        ]\n        dist.all_gather(gathered_peak_memory, peak_memory)\n        peak_memory_by_rank = [float(value.item()) for value in gathered_peak_memory]\n    else:\n        peak_memory_by_rank = [float(peak_memory.item())]\n\n    if is_main:\n        validation_reports: dict[str, dict[str, Any]] = {}\n        validation_predictions: dict[str, pd.DataFrame] = {}\n        for name, validation in validations.items():\n            result = validation_results[name]\n            predictions = build_validation_predictions(\n                validation,\n                validation_caches[name],\n                result,\n                args.max_length,\n            )\n            predictions_filename = f"{name}_validation_predictions.parquet"\n            order_gap = predictions["score_order_gap"]\n            validation_predictions[name] = predictions\n            validation_reports[name] = {\n                "examples": len(predictions),\n                "positive_examples": int(validation_targets[name].sum()),\n                "positive_rate": float(validation_targets[name].mean()),\n                "macro_average_precision": result.macro_average_precision,\n                "overall_average_precision": result.overall_average_precision,\n                "recall_at_precision_0_99": result.recall_at_precision_0_99,\n                "threshold_at_precision_0_99": (\n                    result.threshold_at_precision_0_99\n                ),\n                "roc_auc": result.roc_auc,\n                "log_loss": result.log_loss,\n                "per_category_average_precision": (\n                    result.per_category_average_precision\n                ),\n                "predictions_file": predictions_filename,\n                "mean_score_order_gap": (\n                    float(order_gap.mean()) if order_gap.notna().any() else None\n                ),\n            }\n        primary_name = (\n            "iid"\n            if "iid" in validation_reports\n            else next(iter(validation_reports), None)\n        )\n        primary = validation_reports.get(primary_name) if primary_name else None\n        report = {\n            "world_size": world_size,\n            "amp_dtype": str(amp_dtype),\n            "tokenization_seconds": tokenization_seconds,\n            "training_seconds": float(elapsed.item()),\n            "validation_seconds": validation_seconds,\n            "validation_seconds_by_split": validation_seconds_by_split,\n            "total_pipeline_seconds": time.perf_counter() - pipeline_started,\n            "training_examples": int(totals[0].item()),\n            "original_training_examples": original_train_pairs,\n            "training_subset": args.train_subset,\n            "training_sampling": args.sampling,\n            "training_loss_weighting": args.loss_weighting,\n            "loss_hook": loss_hook.metadata,\n            "training_unique_coverage_per_epoch": (\n                1.0 if args.sampling == "none" else None\n            ),\n            "training_loss_weight_min": float(training_loss_weights.min()),\n            "training_loss_weight_median": float(\n                np.median(training_loss_weights)\n            ),\n            "training_loss_weight_max": float(training_loss_weights.max()),\n            "training_source_counts": training_source_counts,\n            "training_source_weight_mass": training_source_weight_mass,\n            "primary_validation_split": primary_name,\n            "validation_splits": validation_reports,\n            # Compatibility aliases for older analysis tools. The canonical\n            # multi-split results live under validation_splits.\n            "validation_examples": primary["examples"] if primary else 0,\n            "validation_positive_examples": (\n                primary["positive_examples"] if primary else 0\n            ),\n            "validation_positive_rate": (\n                primary["positive_rate"] if primary else None\n            ),\n            "examples_per_second": float(totals[0].item() / elapsed.item()),\n            "padding_efficiency": float(totals[1].item() / totals[2].item()),\n            "peak_vram_gib_by_rank": peak_memory_by_rank,\n            "macro_average_precision": (\n                primary["macro_average_precision"] if primary else None\n            ),\n            "overall_average_precision": (\n                primary["overall_average_precision"] if primary else None\n            ),\n            "per_category_average_precision": (\n                primary["per_category_average_precision"] if primary else {}\n            ),\n            "validation_predictions_file": (\n                primary["predictions_file"] if primary else None\n            ),\n            "mean_score_order_gap": (\n                primary["mean_score_order_gap"] if primary else None\n            ),\n            "args": vars(args),\n        }\n        args.output_dir.mkdir(parents=True, exist_ok=True)\n        inference_model.save_pretrained(args.output_dir, safe_serialization=True)\n        tokenizer.save_pretrained(args.output_dir)\n        for name, predictions in validation_predictions.items():\n            predictions.to_parquet(\n                args.output_dir / validation_reports[name]["predictions_file"],\n                index=False,\n                compression="zstd",\n            )\n        (args.output_dir / "training_report.json").write_text(\n            json.dumps(report, default=str, ensure_ascii=False, indent=2),\n            encoding="utf-8",\n        )\n        (args.output_dir / "training_config.json").write_text(\n            args.config.read_text(encoding="utf-8"), encoding="utf-8"\n        )\n        print(json.dumps(report, default=str, ensure_ascii=False, indent=2), flush=True)\n        print(f"Saved cross-encoder to {args.output_dir}", flush=True)\n    if distributed:\n        dist.barrier()\n        dist.destroy_process_group()\n\n\nif __name__ == "__main__":\n    main()\n', 'src/__init__.py': '"""Inference package for the product matching submission."""\n\n', 'src/cross_encoder_training.py': 'from __future__ import annotations\n\nimport hashlib\nimport json\nimport time\nfrom dataclasses import dataclass\nfrom itertools import chain\nfrom pathlib import Path\nfrom typing import Any, Sequence\n\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom torch.utils.data import Dataset\n\n\nJINA_LBNL_DOC_TOKEN = "<|embed_token|>"\nJINA_LBNL_QUERY_TOKEN = "<|rerank_token|>"\nJINA_LBNL_PREFIX = (\n    "<|im_start|>system\\n"\n    "You are a search relevance expert who can determine a ranking of the passages "\n    "based on how relevant they are to the query. If the query is a question, how "\n    "relevant a passage is depends on how well it answers the question. If not, try "\n    "to analyze the intent of the query and assess how well each passage satisfies "\n    "the intent. If an instruction is provided, you should follow the instruction "\n    "when determining the ranking.<|im_end|>\\n<|im_start|>user\\n"\n    "I will provide you with 1 passages, each indicated by a numerical identifier. "\n    "Rank the passages based on their relevance to query: "\n)\nJINA_LBNL_QUERY_TO_DOCUMENT = \'\\n<passage id="0">\\n\'\nJINA_LBNL_DOCUMENT_TO_QUERY = (\n    f"{JINA_LBNL_DOC_TOKEN}\\n</passage>\\n<query>\\n"\n)\nJINA_LBNL_SUFFIX = (\n    f"{JINA_LBNL_QUERY_TOKEN}\\n</query><|im_end|>\\n"\n    "<|im_start|>assistant\\n<think>\\n\\n</think>\\n\\n"\n)\n\n\ndef _frame_fingerprint(frame: pd.DataFrame, configuration: dict[str, Any]) -> str:\n    digest = hashlib.sha256(\n        json.dumps(configuration, ensure_ascii=False, sort_keys=True).encode("utf-8")\n    )\n    columns = ["id1", "id2", "target", "product_text_1", "product_text_2"]\n    row_hashes = pd.util.hash_pandas_object(frame[columns], index=False).to_numpy()\n    digest.update(row_hashes.tobytes())\n    return digest.hexdigest()[:20]\n\n\n@dataclass(frozen=True)\nclass PairTokenCache:\n    directory: Path\n    forward_tokens: np.ndarray\n    forward_offsets: np.ndarray\n    reverse_tokens: np.ndarray\n    reverse_offsets: np.ndarray\n    forward_token_types: np.ndarray | None = None\n    reverse_token_types: np.ndarray | None = None\n\n    @classmethod\n    def load(cls, directory: Path) -> "PairTokenCache":\n        metadata = json.loads((directory / "metadata.json").read_text(encoding="utf-8"))\n        has_token_types = bool(metadata.get("has_token_type_ids"))\n        return cls(\n            directory=directory,\n            forward_tokens=np.load(directory / "forward_tokens.npy", mmap_mode="r"),\n            forward_offsets=np.load(directory / "forward_offsets.npy", mmap_mode="r"),\n            reverse_tokens=np.load(directory / "reverse_tokens.npy", mmap_mode="r"),\n            reverse_offsets=np.load(directory / "reverse_offsets.npy", mmap_mode="r"),\n            forward_token_types=(\n                np.load(directory / "forward_token_types.npy", mmap_mode="r")\n                if has_token_types\n                else None\n            ),\n            reverse_token_types=(\n                np.load(directory / "reverse_token_types.npy", mmap_mode="r")\n                if has_token_types\n                else None\n            ),\n        )\n\n    @property\n    def size(self) -> int:\n        return len(self.forward_offsets) - 1\n\n    @property\n    def forward_lengths(self) -> np.ndarray:\n        return np.diff(self.forward_offsets)\n\n    @property\n    def reverse_lengths(self) -> np.ndarray:\n        return np.diff(self.reverse_offsets)\n\n    @property\n    def has_token_type_ids(self) -> bool:\n        return self.forward_token_types is not None\n\n    def sequence(self, index: int, reverse: bool = False) -> dict[str, np.ndarray]:\n        tokens = self.reverse_tokens if reverse else self.forward_tokens\n        offsets = self.reverse_offsets if reverse else self.forward_offsets\n        start, end = int(offsets[index]), int(offsets[index + 1])\n        result = {"input_ids": tokens[start:end]}\n        token_types = self.reverse_token_types if reverse else self.forward_token_types\n        if token_types is not None:\n            result["token_type_ids"] = token_types[start:end]\n        return result\n\ndef _cache_is_complete(directory: Path, configuration: dict[str, Any], rows: int) -> bool:\n    metadata_path = directory / "metadata.json"\n    if not metadata_path.is_file():\n        return False\n    metadata = json.loads(metadata_path.read_text(encoding="utf-8"))\n    required = [\n        directory / "forward_tokens.npy",\n        directory / "forward_offsets.npy",\n        directory / "reverse_tokens.npy",\n        directory / "reverse_offsets.npy",\n    ]\n    if metadata.get("has_token_type_ids"):\n        required.extend(\n            [\n                directory / "forward_token_types.npy",\n                directory / "reverse_token_types.npy",\n            ]\n        )\n    return (\n        metadata.get("configuration") == configuration\n        and metadata.get("rows") == rows\n        and all(path.is_file() for path in required)\n    )\n\n\ndef _jina_lbnl_product_prefixes(\n    query_tokens: Sequence[int],\n    document_tokens: Sequence[int],\n    budget: int,\n) -> tuple[Sequence[int], Sequence[int]]:\n    """Allocate a fair budget when the LBNL prompt repeats the query twice."""\n    query_keep = min(len(query_tokens), budget // 4)\n    document_keep = min(len(document_tokens), budget // 2)\n    remaining = budget - 2 * query_keep - document_keep\n\n    if remaining and document_keep == len(document_tokens):\n        extra_query = min(len(query_tokens) - query_keep, remaining // 2)\n        query_keep += extra_query\n        remaining -= 2 * extra_query\n    if remaining and query_keep == len(query_tokens):\n        extra_document = min(len(document_tokens) - document_keep, remaining)\n        document_keep += extra_document\n        remaining -= extra_document\n    if remaining >= 2:\n        extra_query = min(len(query_tokens) - query_keep, remaining // 2)\n        query_keep += extra_query\n        remaining -= 2 * extra_query\n    if remaining:\n        document_keep += min(len(document_tokens) - document_keep, remaining)\n    return query_tokens[:query_keep], document_tokens[:document_keep]\n\n\ndef _jina_lbnl_static_tokens(tokenizer: Any) -> tuple[list[list[int]], int, int, int]:\n    pieces = [\n        tokenizer.encode(JINA_LBNL_PREFIX, add_special_tokens=False),\n        tokenizer.encode(JINA_LBNL_QUERY_TO_DOCUMENT, add_special_tokens=False),\n        tokenizer.encode(JINA_LBNL_DOCUMENT_TO_QUERY, add_special_tokens=False),\n        tokenizer.encode(JINA_LBNL_SUFFIX, add_special_tokens=False),\n    ]\n    doc_token_id = int(tokenizer.convert_tokens_to_ids(JINA_LBNL_DOC_TOKEN))\n    query_token_id = int(tokenizer.convert_tokens_to_ids(JINA_LBNL_QUERY_TOKEN))\n    if doc_token_id < 0 or query_token_id < 0 or doc_token_id == query_token_id:\n        raise ValueError("Jina LBNL tokenizer does not define distinct reranker tokens")\n    static = list(chain.from_iterable(pieces))\n    if static.count(doc_token_id) != 1 or static.count(query_token_id) != 1:\n        raise ValueError(\n            "Jina LBNL prompt must contain exactly one document and one query marker"\n        )\n    return pieces, doc_token_id, query_token_id, len(static)\n\n\ndef _jina_lbnl_sequence(\n    query_tokens: Sequence[int],\n    document_tokens: Sequence[int],\n    static_pieces: Sequence[Sequence[int]],\n    product_budget: int,\n) -> list[int]:\n    query, document = _jina_lbnl_product_prefixes(\n        query_tokens, document_tokens, product_budget\n    )\n    return list(\n        chain(\n            static_pieces[0],\n            query,\n            static_pieces[1],\n            document,\n            static_pieces[2],\n            query,\n            static_pieces[3],\n        )\n    )\n\n\ndef build_pair_token_cache(\n    frame: pd.DataFrame,\n    tokenizer: Any,\n    cache_root: Path,\n    split_name: str,\n    model_name: str,\n    max_length: int,\n    tokenization_batch_size: int = 512,\n    log_every: int = 50,\n    pair_format: str = "cross_encoder",\n) -> PairTokenCache:\n    """Tokenize both pair orientations once and store compact mmap arrays."""\n    if pair_format not in {"cross_encoder", "jina_lbnl"}:\n        raise ValueError(f"Unknown pair tokenization format: {pair_format!r}")\n    configuration = {\n        "version": 1,\n        "model": model_name,\n        "tokenizer_class": type(tokenizer).__name__,\n        "tokenizer_size": len(tokenizer) if hasattr(tokenizer, "__len__") else None,\n        "max_length": max_length,\n    }\n    if pair_format == "jina_lbnl":\n        configuration.update(version=2, pair_format=pair_format)\n    fingerprint = _frame_fingerprint(frame, configuration)\n    directory = cache_root / f"{split_name}-{fingerprint}"\n    if _cache_is_complete(directory, configuration, len(frame)):\n        print(json.dumps({"token_cache_reused": str(directory), "rows": len(frame)}), flush=True)\n        return PairTokenCache.load(directory)\n    directory.mkdir(parents=True, exist_ok=True)\n\n    forward_offsets = np.zeros(len(frame) + 1, dtype=np.int64)\n    reverse_offsets = np.zeros(len(frame) + 1, dtype=np.int64)\n    forward_chunks: list[np.ndarray] = []\n    reverse_chunks: list[np.ndarray] = []\n    forward_type_chunks: list[np.ndarray] = []\n    reverse_type_chunks: list[np.ndarray] = []\n    forward_position = reverse_position = 0\n    has_token_types: bool | None = None\n    started = time.perf_counter()\n    jina_static: list[list[int]] | None = None\n    jina_doc_token_id = jina_query_token_id = -1\n    product_budget = None\n    if pair_format == "jina_lbnl":\n        (\n            jina_static,\n            jina_doc_token_id,\n            jina_query_token_id,\n            static_length,\n        ) = _jina_lbnl_static_tokens(tokenizer)\n        product_budget = max_length - static_length\n        if product_budget < 16:\n            raise ValueError(\n                f"max_length={max_length} leaves only {product_budget} product tokens "\n                "for the Jina LBNL prompt"\n            )\n\n    for batch_number, start in enumerate(\n        range(0, len(frame), tokenization_batch_size), start=1\n    ):\n        part = frame.iloc[start : start + tokenization_batch_size]\n        first_texts = part["product_text_1"].astype(str).tolist()\n        second_texts = part["product_text_2"].astype(str).tolist()\n        part_size = len(part)\n        if pair_format == "cross_encoder":\n            encoded = tokenizer(\n                first_texts + second_texts,\n                second_texts + first_texts,\n                add_special_tokens=True,\n                padding=False,\n                truncation="longest_first",\n                max_length=max_length,\n                return_attention_mask=False,\n            )\n            forward_sequences = encoded["input_ids"][:part_size]\n            reverse_sequences = encoded["input_ids"][part_size:]\n            current_has_token_types = "token_type_ids" in encoded\n        else:\n            assert jina_static is not None and product_budget is not None\n            sanitized = [\n                text.replace(JINA_LBNL_DOC_TOKEN, "").replace(\n                    JINA_LBNL_QUERY_TOKEN, ""\n                )\n                for text in first_texts + second_texts\n            ]\n            product_tokens = tokenizer(\n                sanitized,\n                add_special_tokens=False,\n                padding=False,\n                truncation=False,\n                return_attention_mask=False,\n            )["input_ids"]\n            first_tokens = product_tokens[:part_size]\n            second_tokens = product_tokens[part_size:]\n            forward_sequences = [\n                _jina_lbnl_sequence(first, second, jina_static, product_budget)\n                for first, second in zip(first_tokens, second_tokens)\n            ]\n            reverse_sequences = [\n                _jina_lbnl_sequence(second, first, jina_static, product_budget)\n                for first, second in zip(first_tokens, second_tokens)\n            ]\n            for sequence in chain(forward_sequences, reverse_sequences):\n                if len(sequence) > max_length:\n                    raise RuntimeError("Jina LBNL token budget exceeded max_length")\n                if (\n                    sequence.count(jina_doc_token_id) != 1\n                    or sequence.count(jina_query_token_id) != 1\n                ):\n                    raise RuntimeError("Jina LBNL reranker marker was lost during tokenization")\n            encoded = {}\n            current_has_token_types = False\n        if has_token_types is None:\n            has_token_types = current_has_token_types\n        elif has_token_types != current_has_token_types:\n            raise RuntimeError("Tokenizer changed token_type_ids behavior between batches")\n\n        for offset, (forward, reverse) in enumerate(\n            zip(forward_sequences, reverse_sequences), start=1\n        ):\n            forward_position += len(forward)\n            reverse_position += len(reverse)\n            forward_offsets[start + offset] = forward_position\n            reverse_offsets[start + offset] = reverse_position\n        forward_chunks.append(\n            np.fromiter(chain.from_iterable(forward_sequences), dtype=np.int32)\n        )\n        reverse_chunks.append(\n            np.fromiter(chain.from_iterable(reverse_sequences), dtype=np.int32)\n        )\n        if current_has_token_types:\n            token_types = encoded["token_type_ids"]\n            forward_type_chunks.append(\n                np.fromiter(chain.from_iterable(token_types[:part_size]), dtype=np.int32)\n            )\n            reverse_type_chunks.append(\n                np.fromiter(chain.from_iterable(token_types[part_size:]), dtype=np.int32)\n            )\n\n        processed = start + part_size\n        if batch_number % log_every == 0 or processed == len(frame):\n            elapsed = time.perf_counter() - started\n            print(\n                json.dumps(\n                    {\n                        "tokenizing_split": split_name,\n                        "rows": processed,\n                        "total_rows": len(frame),\n                        "pair_orientations_per_second": 2 * processed / elapsed,\n                        "elapsed_seconds": elapsed,\n                    }\n                ),\n                flush=True,\n            )\n\n    empty = np.empty(0, dtype=np.int32)\n    np.save(\n        directory / "forward_tokens.npy",\n        np.concatenate(forward_chunks) if forward_chunks else empty,\n    )\n    np.save(directory / "forward_offsets.npy", forward_offsets)\n    np.save(\n        directory / "reverse_tokens.npy",\n        np.concatenate(reverse_chunks) if reverse_chunks else empty,\n    )\n    np.save(directory / "reverse_offsets.npy", reverse_offsets)\n    if has_token_types:\n        np.save(directory / "forward_token_types.npy", np.concatenate(forward_type_chunks))\n        np.save(directory / "reverse_token_types.npy", np.concatenate(reverse_type_chunks))\n\n    metadata = {\n        "configuration": configuration,\n        "rows": len(frame),\n        "has_token_type_ids": bool(has_token_types),\n        "elapsed_seconds": time.perf_counter() - started,\n        "forward_tokens": int(forward_position),\n        "reverse_tokens": int(reverse_position),\n        "product_token_budget": product_budget,\n    }\n    (directory / "metadata.json").write_text(\n        json.dumps(metadata, ensure_ascii=False, indent=2), encoding="utf-8"\n    )\n    print(json.dumps({"token_cache": str(directory), **metadata}), flush=True)\n    return PairTokenCache.load(directory)\n\n\nclass CrossEncoderPairDataset(Dataset[dict[str, Any]]):\n    def __init__(\n        self,\n        cache: PairTokenCache,\n        targets: Sequence[float],\n        sample_weights: Sequence[float] | None = None,\n    ) -> None:\n        if cache.size != len(targets):\n            raise ValueError("Token cache and target lengths differ")\n        self.cache = cache\n        self.targets = np.asarray(targets, dtype=np.float32)\n        self.sample_weights = (\n            np.ones(len(targets), dtype=np.float32)\n            if sample_weights is None\n            else np.asarray(sample_weights, dtype=np.float32)\n        )\n\n    def __len__(self) -> int:\n        return len(self.targets)\n\n    def __getitem__(self, key: int | tuple[int, bool]) -> dict[str, Any]:\n        if isinstance(key, tuple):\n            index, reverse = key\n        else:\n            index, reverse = key, False\n        return {\n            **self.cache.sequence(index, reverse=reverse),\n            "target": self.targets[index],\n            "sample_weight": self.sample_weights[index],\n            "pair_index": index,\n            "reverse": reverse,\n        }\n\n\n@dataclass(frozen=True)\nclass CrossEncoderBatchCollator:\n    pad_token_id: int\n\n    def __call__(self, rows: list[dict[str, Any]]) -> dict[str, torch.Tensor]:\n        max_length = max(len(row["input_ids"]) for row in rows)\n        input_ids = torch.full(\n            (len(rows), max_length), self.pad_token_id, dtype=torch.long\n        )\n        attention_mask = torch.zeros((len(rows), max_length), dtype=torch.long)\n        has_token_types = "token_type_ids" in rows[0]\n        token_type_ids = (\n            torch.zeros((len(rows), max_length), dtype=torch.long)\n            if has_token_types\n            else None\n        )\n        for row_index, row in enumerate(rows):\n            length = len(row["input_ids"])\n            input_ids[row_index, :length] = torch.from_numpy(\n                np.asarray(row["input_ids"]).copy()\n            ).to(dtype=torch.long)\n            attention_mask[row_index, :length] = 1\n            if token_type_ids is not None:\n                token_type_ids[row_index, :length] = torch.from_numpy(\n                    np.asarray(row["token_type_ids"]).copy()\n                ).to(dtype=torch.long)\n        batch = {\n            "input_ids": input_ids,\n            "attention_mask": attention_mask,\n            "targets": torch.tensor([row["target"] for row in rows], dtype=torch.float32),\n            "sample_weights": torch.tensor(\n                [row["sample_weight"] for row in rows], dtype=torch.float32\n            ),\n            "pair_indices": torch.tensor(\n                [row["pair_index"] for row in rows], dtype=torch.long\n            ),\n            "orientations": torch.tensor(\n                [row["reverse"] for row in rows], dtype=torch.bool\n            ),\n        }\n        if token_type_ids is not None:\n            batch["token_type_ids"] = token_type_ids\n        return batch\n', 'src/cross_encoder_experiment_hooks.py': 'from __future__ import annotations\n\nimport hashlib\nimport importlib.util\nimport re\nimport sys\nfrom dataclasses import dataclass\nfrom pathlib import Path\nfrom types import ModuleType\nfrom typing import Any, Mapping\n\nimport torch\nimport torch.nn.functional as F\n\n\n_METRIC_NAME = re.compile(r"[a-zA-Z][a-zA-Z0-9_.-]*")\n\n\ndef _default_compute_loss(\n    *,\n    logits: torch.Tensor,\n    targets: torch.Tensor,\n    sample_weights: torch.Tensor,\n    **_: Any,\n) -> dict[str, torch.Tensor]:\n    per_example = F.binary_cross_entropy_with_logits(\n        logits.float(), targets, reduction="none"\n    )\n    loss = (per_example * sample_weights).sum() / sample_weights.sum()\n    return {"loss": loss, "bce": loss.detach()}\n\n\ndef _file_sha256(path: Path) -> str:\n    digest = hashlib.sha256()\n    with path.open("rb") as source:\n        for chunk in iter(lambda: source.read(1024 * 1024), b""):\n            digest.update(chunk)\n    return digest.hexdigest()\n\n\n@dataclass(frozen=True)\nclass LoadedLossHook:\n    """A small, explicit extension point for controlled loss ablations."""\n\n    name: str\n    path: Path | None\n    sha256: str | None\n    module: ModuleType | None\n\n    @property\n    def metadata(self) -> dict[str, str | None]:\n        return {\n            "name": self.name,\n            "path": str(self.path) if self.path is not None else None,\n            "sha256": self.sha256,\n        }\n\n    def initialize(self, **context: Any) -> None:\n        if self.module is None:\n            return\n        initialize = getattr(self.module, "initialize_loss", None)\n        if initialize is not None:\n            initialize(**context)\n\n    def compute(self, **context: Any) -> tuple[torch.Tensor, dict[str, torch.Tensor]]:\n        compute = (\n            _default_compute_loss\n            if self.module is None\n            else getattr(self.module, "compute_loss")\n        )\n        result = compute(**context)\n        if isinstance(result, torch.Tensor):\n            loss = result\n            raw_metrics: Mapping[str, Any] = {}\n        elif isinstance(result, Mapping):\n            if "loss" not in result:\n                raise ValueError("Loss hook mapping must contain a \'loss\' tensor")\n            loss = result["loss"]\n            raw_metrics = {key: value for key, value in result.items() if key != "loss"}\n        else:\n            raise TypeError("Loss hook must return a scalar tensor or a mapping with \'loss\'")\n        if not isinstance(loss, torch.Tensor) or loss.ndim != 0:\n            raise ValueError("Loss hook \'loss\' must be a scalar torch.Tensor")\n        metrics: dict[str, torch.Tensor] = {}\n        for name, value in raw_metrics.items():\n            if not isinstance(name, str) or _METRIC_NAME.fullmatch(name) is None:\n                raise ValueError(f"Invalid loss metric name: {name!r}")\n            metric = value if isinstance(value, torch.Tensor) else loss.new_tensor(value)\n            if metric.numel() != 1:\n                raise ValueError(f"Loss metric {name!r} must contain one value")\n            metrics[name] = metric.reshape(()).detach()\n        return loss, metrics\n\n\ndef load_loss_hook(path: Path | None) -> LoadedLossHook:\n    if path is None:\n        return LoadedLossHook(\n            name="weighted_bce",\n            path=None,\n            sha256=None,\n            module=None,\n        )\n    resolved = path.resolve()\n    if not resolved.is_file():\n        raise FileNotFoundError(f"Loss hook does not exist: {resolved}")\n    source_hash = _file_sha256(resolved)\n    module_name = f"product_matching_loss_hook_{source_hash[:16]}"\n    spec = importlib.util.spec_from_file_location(module_name, resolved)\n    if spec is None or spec.loader is None:\n        raise ImportError(f"Cannot load loss hook: {resolved}")\n    module = importlib.util.module_from_spec(spec)\n    sys.modules[module_name] = module\n    try:\n        spec.loader.exec_module(module)\n    except Exception:\n        sys.modules.pop(module_name, None)\n        raise\n    if not callable(getattr(module, "compute_loss", None)):\n        raise TypeError(f"Loss hook must define compute_loss(...): {resolved}")\n    initialize = getattr(module, "initialize_loss", None)\n    if initialize is not None and not callable(initialize):\n        raise TypeError("Loss hook initialize_loss must be callable when provided")\n    return LoadedLossHook(\n        name=resolved.stem,\n        path=resolved,\n        sha256=source_hash,\n        module=module,\n    )\n', 'src/data_pipeline.py': 'from __future__ import annotations\n\nimport json\nimport re\nfrom dataclasses import dataclass\nfrom typing import Any, Iterable\n\nimport numpy as np\nimport pandas as pd\n\n\n# The order is intentionally semantic rather than frequency-based. A substring\n# match covers variants such as "партномер (артикул производителя)".\nPRIORITY_KEY_PARTS = (\n    "бренд",\n    "brand",\n    "модель",\n    "model",\n    "артикул",\n    "партномер",\n    "part number",\n    "sku",\n    "код товара",\n    "тип",\n    "вид",\n    "размер",\n    "объем",\n    "объём",\n    "вес",\n    "цвет",\n    "материал",\n    "комплектац",\n)\n\n_SPACE = re.compile(r"\\s+")\n\n\ndef clean_field(value: Any) -> str:\n    """Normalize whitespace without damaging model numbers or punctuation."""\n    if value is None:\n        return ""\n    if isinstance(value, (dict, list)):\n        value = json.dumps(value, ensure_ascii=False, sort_keys=True)\n    return _SPACE.sub(" ", str(value)).strip()\n\n\ndef _key_priority(key: str) -> tuple[int, str]:\n    normalized = clean_field(key).casefold()\n    for rank, part in enumerate(PRIORITY_KEY_PARTS):\n        if part in normalized:\n            return rank, normalized\n    return len(PRIORITY_KEY_PARTS), normalized\n\n\ndef serialize_attributes(raw_attributes: str, max_chars: int | None = 6000) -> str:\n    """Convert JSON attributes into deterministic, readable key/value lines.\n\n    Character truncation is only a storage/safety guard. The final model input\n    must additionally be truncated with its tokenizer.\n    """\n    try:\n        attributes = json.loads(raw_attributes)\n    except (TypeError, json.JSONDecodeError) as error:\n        raise ValueError(f"Invalid attributes JSON: {str(raw_attributes)[:120]}") from error\n    if not isinstance(attributes, dict):\n        raise ValueError("Attributes JSON must contain an object")\n\n    fields: list[str] = []\n    for key, value in sorted(attributes.items(), key=lambda item: _key_priority(item[0])):\n        key_text, value_text = clean_field(key), clean_field(value)\n        if key_text and value_text:\n            fields.append(f"{key_text}: {value_text}")\n    text = "\\n".join(fields)\n    if max_chars is not None and len(text) > max_chars:\n        text = (\n            text[:max_chars].rsplit("\\n", 1)[0].rstrip()\n            + "\\nХарактеристики обрезаны: да"\n        )\n    return text\n\n\ndef serialize_product(row: pd.Series, max_attribute_chars: int | None = 6000) -> str:\n    """Serialize a product as one ``field: value`` record per line.\n\n    Keeping the category, name and attributes in the same flat representation\n    makes truncation predictable: identifiers and other high-priority\n    attributes are emitted first by :func:`serialize_attributes`.\n    """\n    parts = [\n        f"Категория: {clean_field(row[\'category\'])}",\n        f"Название: {clean_field(row[\'name\'])}",\n    ]\n    attributes = serialize_attributes(row["attributes"], max_chars=max_attribute_chars)\n    if attributes:\n        parts.extend(attributes.splitlines())\n    return "\\n".join(parts)\n\n\ndef truncate_tokens(text: str, tokenizer: Any, max_tokens: int) -> str:\n    """Tokenizer-aware truncation for the final per-product budget."""\n    token_ids = tokenizer.encode(text, add_special_tokens=False, truncation=True, max_length=max_tokens)\n    return tokenizer.decode(token_ids, skip_special_tokens=True)\n\n\n@dataclass(frozen=True)\nclass SplitDiagnostics:\n    train_pairs: int\n    validation_pairs: int\n    train_items: int\n    validation_items: int\n    overlapping_items: int\n\n\ndef component_split(\n    matches: pd.DataFrame, validation_fraction: float = 0.15, seed: int = 42\n) -> tuple[pd.DataFrame, pd.DataFrame, SplitDiagnostics]:\n    """Split whole connected components so no product leaks across splits."""\n    if not 0.0 < validation_fraction < 1.0:\n        raise ValueError("validation_fraction must be between 0 and 1")\n\n    all_ids = pd.unique(matches[["id1", "id2"]].to_numpy().reshape(-1))\n    positions = pd.Series(np.arange(len(all_ids), dtype=np.int64), index=all_ids)\n    left = positions.loc[matches["id1"]].to_numpy()\n    right = positions.loc[matches["id2"]].to_numpy()\n    parent = np.arange(len(all_ids), dtype=np.int64)\n    size = np.ones(len(all_ids), dtype=np.int64)\n\n    def find(node: int) -> int:\n        while parent[node] != node:\n            parent[node] = parent[parent[node]]\n            node = int(parent[node])\n        return node\n\n    for first, second in zip(left, right):\n        root1, root2 = find(int(first)), find(int(second))\n        if root1 == root2:\n            continue\n        if size[root1] < size[root2]:\n            root1, root2 = root2, root1\n        parent[root2] = root1\n        size[root1] += size[root2]\n\n    component = np.fromiter((find(int(node)) for node in left), dtype=np.int64, count=len(left))\n    unique_components = np.unique(component)\n    rng = np.random.default_rng(seed)\n    validation_components = unique_components[\n        rng.random(len(unique_components)) < validation_fraction\n    ]\n    is_validation = np.isin(component, validation_components)\n    train = matches.loc[~is_validation].reset_index(drop=True)\n    validation = matches.loc[is_validation].reset_index(drop=True)\n\n    train_ids = set(train["id1"]) | set(train["id2"])\n    validation_ids = set(validation["id1"]) | set(validation["id2"])\n    diagnostics = SplitDiagnostics(\n        train_pairs=len(train),\n        validation_pairs=len(validation),\n        train_items=len(train_ids),\n        validation_items=len(validation_ids),\n        overlapping_items=len(train_ids & validation_ids),\n    )\n    if diagnostics.overlapping_items:\n        raise RuntimeError("Internal error: item leakage in component split")\n    return train, validation, diagnostics\n\n\ndef attach_item_fields(\n    pairs: pd.DataFrame, items: pd.DataFrame, fields: Iterable[str] = ("name", "category")\n) -> pd.DataFrame:\n    fields = list(fields)\n    lookup = items.set_index("id", verify_integrity=True)[fields]\n    left = lookup.reindex(pairs["id1"].to_numpy()).add_suffix("_1")\n    right = lookup.reindex(pairs["id2"].to_numpy()).add_suffix("_2")\n    left.index, right.index = pairs.index, pairs.index\n    result = pd.concat([pairs, left, right], axis=1)\n    if result[[f"{field}_{side}" for field in fields for side in (1, 2)]].isna().any().any():\n        raise ValueError("Some pair ids are absent from items")\n    return result\n', 'src/experiment_protocol.py': '"""Shared, dependency-light helpers for the frozen validation protocol."""\n\nfrom __future__ import annotations\n\nfrom pathlib import Path\nfrom typing import Sequence\n\n\ndef validation_split_paths(\n    prepared_dir: Path,\n    specs: Sequence[str],\n) -> dict[str, Path]:\n    values = specs or ["iid=val_pairs.parquet"]\n    result: dict[str, Path] = {}\n    for spec in values:\n        name, separator, raw_path = spec.partition("=")\n        name = name.strip().lower()\n        raw_path = raw_path.strip()\n        if not separator or not name or not raw_path:\n            raise ValueError(\n                f"Invalid --validation-split {spec!r}; expected NAME=PATH"\n            )\n        if not name.replace("_", "").isalnum():\n            raise ValueError(\n                f"Validation split name must contain only letters, digits and underscores: {name!r}"\n            )\n        if name in result:\n            raise ValueError(f"Duplicate validation split name: {name!r}")\n        path = Path(raw_path)\n        result[name] = path if path.is_absolute() else prepared_dir / path\n    return result\n', 'src/pair_features.py': 'from __future__ import annotations\n\nfrom typing import Any, Sequence\n\nimport numpy as np\nimport pandas as pd\nfrom sklearn.feature_extraction.text import HashingVectorizer\n\n\ndef extract_product_names(product_texts: Sequence[Any]) -> pd.Series:\n    """Extract the serialized ``Название`` line without depending on raw items."""\n    texts = pd.Series(product_texts, dtype="string")\n    return (\n        texts.str.extract(r"(?mi)^Название:\\s*(.*)$", expand=False)\n        .fillna("")\n        .str.casefold()\n        .str.replace(r"\\s+", " ", regex=True)\n        .str.strip()\n    )\n\n\ndef name_ngram_cosine(\n    frame: pd.DataFrame,\n    *,\n    batch_size: int = 8192,\n    n_features: int = 2**18,\n) -> np.ndarray:\n    """Compute stateless char 3-5-gram cosine similarity for each product pair."""\n    first = extract_product_names(frame["product_text_1"])\n    second = extract_product_names(frame["product_text_2"])\n    vectorizer = HashingVectorizer(\n        analyzer="char_wb",\n        ngram_range=(3, 5),\n        n_features=n_features,\n        alternate_sign=False,\n        norm="l2",\n        lowercase=False,\n    )\n    similarities = np.empty(len(frame), dtype=np.float32)\n    for start in range(0, len(frame), batch_size):\n        stop = min(len(frame), start + batch_size)\n        first_vectors = vectorizer.transform(first.iloc[start:stop])\n        second_vectors = vectorizer.transform(second.iloc[start:stop])\n        similarities[start:stop] = np.asarray(\n            first_vectors.multiply(second_vectors).sum(axis=1)\n        ).ravel()\n    return np.clip(similarities, 0.0, 1.0)\n\n\ndef category_label_downsample(\n    frame: pd.DataFrame,\n    *,\n    category_column: str = "category_1",\n    target_column: str = "target",\n    seed: int = 42,\n) -> pd.DataFrame:\n    """Downsample each category\'s majority label without balancing categories.\n\n    Every minority row is retained. Categories therefore keep different sizes,\n    determined by their available minority examples, and no row is repeated.\n    """\n    if category_column not in frame or target_column not in frame:\n        raise ValueError("Training frame is missing category or target columns")\n    if not frame[target_column].isin([0.0, 1.0]).all():\n        raise ValueError("Category-label downsampling requires binary targets")\n\n    selected: list[np.ndarray] = []\n    rng = np.random.default_rng(seed)\n    labels = frame[target_column].astype(np.int8)\n    for _, category_positions in frame.groupby(\n        category_column, sort=True, dropna=False\n    ).indices.items():\n        category_positions = np.asarray(category_positions, dtype=np.int64)\n        category_labels = labels.iloc[category_positions].to_numpy()\n        negative_positions = category_positions[category_labels == 0]\n        positive_positions = category_positions[category_labels == 1]\n        if not len(negative_positions) or not len(positive_positions):\n            raise ValueError("Every category must contain both target labels")\n        size = min(len(negative_positions), len(positive_positions))\n        selected.extend(\n            [\n                rng.choice(negative_positions, size=size, replace=False),\n                rng.choice(positive_positions, size=size, replace=False),\n            ]\n        )\n    positions = np.concatenate(selected)\n    rng.shuffle(positions)\n    return frame.iloc[positions].reset_index(drop=True)\n\n\ndef build_training_loss_weights(\n    categories: Sequence[Any],\n    targets: Sequence[float],\n    *,\n    mode: str = "none",\n    lexical_similarities: Sequence[float] | None = None,\n    lexical_hard_negative_strength: float = 0.0,\n) -> np.ndarray:\n    """Build moderate loss weights while retaining every training example.\n\n    ``category_label_sqrt`` is deliberately softer than inverse-frequency\n    resampling: a minority example gets a square-root frequency correction,\n    while the epoch still contains every row exactly once. Optional lexical\n    weighting only redistributes weight *within* each category\'s negatives, so\n    highly similar hard negatives receive more attention without changing the\n    total category/label mass.\n    """\n    targets_array = np.asarray(targets, dtype=np.float32)\n    data = pd.DataFrame(\n        {\n            "category": pd.Series(categories, dtype="string"),\n            "label": (targets_array >= 0.5).astype(np.int8),\n        }\n    )\n    if mode == "none":\n        weights = np.ones(len(data), dtype=np.float64)\n    elif mode == "category_label_sqrt":\n        counts = data.groupby(["category", "label"], dropna=False)[\n            "label"\n        ].transform("size")\n        weights = 1.0 / np.sqrt(counts.to_numpy(dtype=np.float64))\n    else:\n        raise ValueError(f"Unknown loss-weighting mode: {mode}")\n\n    if lexical_hard_negative_strength:\n        if lexical_similarities is None:\n            raise ValueError("Lexical similarities are required for hard-negative weights")\n        similarities = np.asarray(lexical_similarities, dtype=np.float64)\n        if len(similarities) != len(data):\n            raise ValueError("Lexical similarities and targets have different lengths")\n        if not np.isfinite(similarities).all():\n            raise ValueError("Lexical similarities must be finite")\n        modifier = np.ones(len(data), dtype=np.float64)\n        negatives = data["label"].eq(0).to_numpy()\n        modifier[negatives] += (\n            lexical_hard_negative_strength\n            * np.clip(similarities[negatives], 0.0, 1.0)\n        )\n        # Preserve each negative category\'s total mass and only redistribute it\n        # from easy to lexically confusing pairs.\n        negative_groups = data.loc[negatives].groupby("category", dropna=False).indices\n        negative_positions = np.flatnonzero(negatives)\n        for positions in negative_groups.values():\n            absolute_positions = negative_positions[np.asarray(positions)]\n            modifier[absolute_positions] /= modifier[absolute_positions].mean()\n        weights *= modifier\n\n    weights /= weights.mean()\n    return weights.astype(np.float32)\n', 'src/qwen_reranker.py': 'from __future__ import annotations\n\nfrom dataclasses import dataclass\nfrom typing import Any\n\nimport torch\n\n\nINSTRUCTION = (\n    "Determine whether Query and Document are listings of exactly the same marketplace "\n    "product and the same variant. Ignore wording and attribute-schema differences. "\n    "Answer no if the model, part number or SKU, size, color, volume, quantity, material, "\n    "or bundle configuration differs."\n)\nPREFIX = (\n    \'<|im_start|>system\\nJudge whether the Document meets the requirements based on the Query \'\n    \'and the Instruct provided. Note that the answer can only be "yes" or "no".\'\n    \'<|im_end|>\\n<|im_start|>user\\n\'\n)\nSUFFIX = "<|im_end|>\\n<|im_start|>assistant\\n<think>\\n\\n</think>\\n\\n"\n\n\ndef preferred_cuda_dtype() -> torch.dtype:\n    """Use bf16 on Ampere/Hopper and fp16 on older Kaggle GPUs such as T4."""\n    major, _ = torch.cuda.get_device_capability()\n    return torch.bfloat16 if major >= 8 else torch.float16\n\n\ndef format_pair(product1: str, product2: str, instruction: str = INSTRUCTION) -> str:\n    """Format two line-oriented product records for the original reranker prompt."""\n    return (\n        f"<Instruct>: {instruction}\\n"\n        f"<Query>:\\n{product1}\\n"\n        f"<Document>:\\n{product2}"\n    )\n\n\n@dataclass\nclass QwenBatchCollator:\n    tokenizer: Any\n    max_length: int = 256\n    include_labels: bool = False\n\n    def __post_init__(self) -> None:\n        self.prefix_ids = self.tokenizer.encode(PREFIX, add_special_tokens=False)\n        self.suffix_ids = self.tokenizer.encode(SUFFIX, add_special_tokens=False)\n        self.yes_id = self.tokenizer("yes", add_special_tokens=False).input_ids[0]\n        self.no_id = self.tokenizer("no", add_special_tokens=False).input_ids[0]\n        self.tokenizer.padding_side = "left"\n        if self.tokenizer.pad_token_id is None:\n            self.tokenizer.pad_token = self.tokenizer.eos_token\n\n    def __call__(self, rows: list[dict[str, Any]]) -> dict[str, torch.Tensor]:\n        reserved = len(self.prefix_ids) + len(self.suffix_ids)\n        pair_budget = self.max_length - reserved\n        if pair_budget <= 0:\n            raise ValueError("max_length is too small for the Qwen prompt")\n        pair_texts, targets = [], []\n        for row in rows:\n            first = row.get("product_text_1", row.get("text_1", row.get("name_1")))\n            second = row.get("product_text_2", row.get("text_2", row.get("name_2")))\n            if first is None or second is None:\n                raise KeyError(\n                    "Rows must contain product_text_1/product_text_2 or name_1/name_2"\n                )\n            pair_texts.append(format_pair(str(first), str(second)))\n            if self.include_labels:\n                targets.append(float(row["target"]))\n\n        # One batched Rust-tokenizer call is much faster than one Python call per\n        # sample. Training uses a persistent on-disk token cache, while this\n        # collator remains useful for inference and small smoke tests.\n        encoded = self.tokenizer(\n            pair_texts,\n            add_special_tokens=False,\n            truncation=True,\n            max_length=pair_budget,\n            padding=False,\n            return_attention_mask=False,\n        )["input_ids"]\n        sequences = [\n            self.prefix_ids + pair_ids + self.suffix_ids for pair_ids in encoded\n        ]\n\n        batch = self.tokenizer.pad(\n            {"input_ids": sequences}, padding=True, return_tensors="pt"\n        )\n        if self.include_labels:\n            batch["targets"] = torch.tensor(targets, dtype=torch.float32)\n        return batch\n\n\ndef yes_probability(logits: torch.Tensor, yes_id: int, no_id: int) -> torch.Tensor:\n    pair_logits = torch.stack((logits[:, -1, no_id], logits[:, -1, yes_id]), dim=1)\n    return pair_logits.softmax(dim=1)[:, 1]\n', 'src/qwen_training.py': 'from __future__ import annotations\n\nimport hashlib\nimport json\nimport math\nimport time\nfrom dataclasses import dataclass\nfrom itertools import chain\nfrom pathlib import Path\nfrom typing import Any, Iterator, Sequence\n\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom torch.utils.data import Dataset, Sampler\n\nfrom src.qwen_reranker import INSTRUCTION, PREFIX, SUFFIX\n\n\ndef _frame_fingerprint(frame: pd.DataFrame, configuration: dict[str, Any]) -> str:\n    """Hash the texts and IDs so stale token caches cannot be reused silently."""\n    digest = hashlib.sha256(\n        json.dumps(configuration, ensure_ascii=False, sort_keys=True).encode("utf-8")\n    )\n    columns = ["id1", "id2", "target", "product_text_1", "product_text_2"]\n    row_hashes = pd.util.hash_pandas_object(frame[columns], index=False).to_numpy()\n    digest.update(row_hashes.tobytes())\n    return digest.hexdigest()[:20]\n\n\ndef _balanced_prefixes(\n    first: Sequence[int], second: Sequence[int], budget: int\n) -> tuple[Sequence[int], Sequence[int]]:\n    """Keep the beginnings of both products and give unused space to the longer one."""\n    if len(first) + len(second) <= budget:\n        return first, second\n    first_keep = min(len(first), budget // 2)\n    second_keep = min(len(second), budget // 2)\n    remaining = budget - first_keep - second_keep\n    if remaining:\n        add_first = min(len(first) - first_keep, remaining)\n        first_keep += add_first\n        remaining -= add_first\n        second_keep += min(len(second) - second_keep, remaining)\n    return first[:first_keep], second[:second_keep]\n\n\n@dataclass(frozen=True)\nclass TokenCache:\n    directory: Path\n    forward_tokens: np.ndarray\n    forward_offsets: np.ndarray\n    reverse_tokens: np.ndarray\n    reverse_offsets: np.ndarray\n\n    @classmethod\n    def load(cls, directory: Path) -> "TokenCache":\n        return cls(\n            directory=directory,\n            forward_tokens=np.load(directory / "forward_tokens.npy", mmap_mode="r"),\n            forward_offsets=np.load(directory / "forward_offsets.npy", mmap_mode="r"),\n            reverse_tokens=np.load(directory / "reverse_tokens.npy", mmap_mode="r"),\n            reverse_offsets=np.load(directory / "reverse_offsets.npy", mmap_mode="r"),\n        )\n\n    @property\n    def size(self) -> int:\n        return len(self.forward_offsets) - 1\n\n    @property\n    def forward_lengths(self) -> np.ndarray:\n        return np.diff(self.forward_offsets)\n\n    @property\n    def reverse_lengths(self) -> np.ndarray:\n        return np.diff(self.reverse_offsets)\n\n    def sequence(self, index: int, reverse: bool = False) -> np.ndarray:\n        tokens = self.reverse_tokens if reverse else self.forward_tokens\n        offsets = self.reverse_offsets if reverse else self.forward_offsets\n        start, end = int(offsets[index]), int(offsets[index + 1])\n        return tokens[start:end]\n\n\ndef build_token_cache(\n    frame: pd.DataFrame,\n    tokenizer: Any,\n    cache_root: Path,\n    split_name: str,\n    model_name: str,\n    max_length: int,\n    tokenization_batch_size: int = 512,\n) -> TokenCache:\n    """Batched-tokenize both pair orientations and store compact mmap arrays."""\n    configuration = {\n        "version": 2,\n        "model": model_name,\n        "tokenizer_class": type(tokenizer).__name__,\n        "tokenizer_size": len(tokenizer) if hasattr(tokenizer, "__len__") else None,\n        "max_length": max_length,\n        "instruction": INSTRUCTION,\n        "prefix": PREFIX,\n        "suffix": SUFFIX,\n    }\n    fingerprint = _frame_fingerprint(frame, configuration)\n    directory = cache_root / f"{split_name}-{fingerprint}"\n    metadata_path = directory / "metadata.json"\n    required = (\n        metadata_path,\n        directory / "forward_tokens.npy",\n        directory / "forward_offsets.npy",\n        directory / "reverse_tokens.npy",\n        directory / "reverse_offsets.npy",\n    )\n    if all(path.exists() for path in required):\n        cached = json.loads(metadata_path.read_text(encoding="utf-8"))\n        if cached.get("configuration") == configuration and cached.get("rows") == len(frame):\n            return TokenCache.load(directory)\n\n    directory.mkdir(parents=True, exist_ok=True)\n    prefix_ids = tokenizer.encode(PREFIX, add_special_tokens=False)\n    query_ids = tokenizer.encode(\n        f"<Instruct>: {INSTRUCTION}\\n<Query>:\\n", add_special_tokens=False\n    )\n    document_ids = tokenizer.encode("\\n<Document>:\\n", add_special_tokens=False)\n    suffix_ids = tokenizer.encode(SUFFIX, add_special_tokens=False)\n    product_budget = max_length - sum(\n        map(len, (prefix_ids, query_ids, document_ids, suffix_ids))\n    )\n    if product_budget < 16:\n        raise ValueError(\n            f"max_length={max_length} leaves only {product_budget} product tokens"\n        )\n\n    forward_offsets = np.zeros(len(frame) + 1, dtype=np.int64)\n    reverse_offsets = np.zeros(len(frame) + 1, dtype=np.int64)\n    forward_chunks: list[np.ndarray] = []\n    reverse_chunks: list[np.ndarray] = []\n    forward_position = reverse_position = 0\n    started = time.perf_counter()\n\n    for start in range(0, len(frame), tokenization_batch_size):\n        part = frame.iloc[start : start + tokenization_batch_size]\n        first_texts = part["product_text_1"].astype(str).tolist()\n        second_texts = part["product_text_2"].astype(str).tolist()\n        encoded = tokenizer(\n            first_texts + second_texts,\n            add_special_tokens=False,\n            padding=False,\n            truncation=False,\n            return_attention_mask=False,\n        )["input_ids"]\n        first_ids = encoded[: len(part)]\n        second_ids = encoded[len(part) :]\n        forward_sequences: list[list[int]] = []\n        reverse_sequences: list[list[int]] = []\n\n        for offset, (first, second) in enumerate(zip(first_ids, second_ids), start=1):\n            kept_first, kept_second = _balanced_prefixes(first, second, product_budget)\n            forward = list(\n                chain(prefix_ids, query_ids, kept_first, document_ids, kept_second, suffix_ids)\n            )\n            kept_second, kept_first = _balanced_prefixes(second, first, product_budget)\n            reverse = list(\n                chain(prefix_ids, query_ids, kept_second, document_ids, kept_first, suffix_ids)\n            )\n            forward_sequences.append(forward)\n            reverse_sequences.append(reverse)\n            forward_position += len(forward)\n            reverse_position += len(reverse)\n            forward_offsets[start + offset] = forward_position\n            reverse_offsets[start + offset] = reverse_position\n\n        forward_chunks.append(\n            np.fromiter(chain.from_iterable(forward_sequences), dtype=np.int32)\n        )\n        reverse_chunks.append(\n            np.fromiter(chain.from_iterable(reverse_sequences), dtype=np.int32)\n        )\n\n    np.save(\n        directory / "forward_tokens.npy",\n        np.concatenate(forward_chunks) if forward_chunks else np.empty(0, dtype=np.int32),\n    )\n    np.save(directory / "forward_offsets.npy", forward_offsets)\n    np.save(\n        directory / "reverse_tokens.npy",\n        np.concatenate(reverse_chunks) if reverse_chunks else np.empty(0, dtype=np.int32),\n    )\n    np.save(directory / "reverse_offsets.npy", reverse_offsets)\n    metadata = {\n        "configuration": configuration,\n        "rows": len(frame),\n        "product_token_budget": product_budget,\n        "elapsed_seconds": time.perf_counter() - started,\n        "forward_tokens": int(forward_position),\n        "reverse_tokens": int(reverse_position),\n    }\n    metadata_path.write_text(\n        json.dumps(metadata, ensure_ascii=False, indent=2), encoding="utf-8"\n    )\n    print(json.dumps({"token_cache": str(directory), **metadata}, ensure_ascii=False))\n    return TokenCache.load(directory)\n\n\nclass PackedPairDataset(Dataset[dict[str, Any]]):\n    def __init__(\n        self,\n        cache: TokenCache,\n        targets: Sequence[float],\n        sample_weights: Sequence[float] | None = None,\n    ) -> None:\n        if cache.size != len(targets):\n            raise ValueError("Token cache and target lengths differ")\n        self.cache = cache\n        self.targets = np.asarray(targets, dtype=np.float32)\n        self.sample_weights = (\n            np.ones(len(targets), dtype=np.float32)\n            if sample_weights is None\n            else np.asarray(sample_weights, dtype=np.float32)\n        )\n\n    def __len__(self) -> int:\n        return len(self.targets)\n\n    def __getitem__(self, key: int | tuple[int, bool]) -> dict[str, Any]:\n        if isinstance(key, tuple):\n            index, reverse = key\n        else:\n            index, reverse = key, False\n        return {\n            "input_ids": self.cache.sequence(index, reverse=reverse),\n            "target": self.targets[index],\n            "sample_weight": self.sample_weights[index],\n            "pair_index": index,\n            "reverse": reverse,\n        }\n\n\n@dataclass(frozen=True)\nclass PackedBatchCollator:\n    pad_token_id: int\n\n    def __call__(self, rows: list[dict[str, Any]]) -> dict[str, torch.Tensor]:\n        max_length = max(len(row["input_ids"]) for row in rows)\n        input_ids = torch.full(\n            (len(rows), max_length), self.pad_token_id, dtype=torch.long\n        )\n        attention_mask = torch.zeros((len(rows), max_length), dtype=torch.long)\n        for row_index, row in enumerate(rows):\n            sequence = torch.tensor(row["input_ids"], dtype=torch.long)\n            input_ids[row_index, -len(sequence) :] = sequence\n            attention_mask[row_index, -len(sequence) :] = 1\n        return {\n            "input_ids": input_ids,\n            "attention_mask": attention_mask,\n            "targets": torch.tensor([row["target"] for row in rows], dtype=torch.float32),\n            "sample_weights": torch.tensor(\n                [row["sample_weight"] for row in rows], dtype=torch.float32\n            ),\n            "pair_indices": torch.tensor(\n                [row["pair_index"] for row in rows], dtype=torch.long\n            ),\n            "orientations": torch.tensor(\n                [row["reverse"] for row in rows], dtype=torch.bool\n            ),\n        }\n\n\ndef balanced_sampling_weights(\n    categories: Sequence[Any], targets: Sequence[float], mode: str\n) -> np.ndarray | None:\n    if mode == "none":\n        return None\n    data = pd.DataFrame(\n        {"category": pd.Series(categories, dtype="string"), "target": np.asarray(targets)}\n    )\n    group_columns = ["category"]\n    if mode == "category_label":\n        data["label"] = (data["target"] >= 0.5).astype(np.int8)\n        group_columns.append("label")\n    elif mode != "category":\n        raise ValueError(f"Unknown sampling mode: {mode}")\n    counts = data.groupby(group_columns, dropna=False)["target"].transform("size")\n    weights = 1.0 / counts.to_numpy(dtype=np.float64)\n    return weights / weights.sum()\n\n\nclass LengthBucketBatchSampler(Sampler[list[tuple[int, bool]]]):\n    """Balanced DDP sampling followed by local length bucketing."""\n\n    def __init__(\n        self,\n        forward_lengths: Sequence[int],\n        reverse_lengths: Sequence[int],\n        batch_size: int,\n        rank: int = 0,\n        world_size: int = 1,\n        weights: np.ndarray | None = None,\n        bucket_size_multiplier: int = 50,\n        seed: int = 42,\n    ) -> None:\n        self.forward_lengths = np.asarray(forward_lengths)\n        self.reverse_lengths = np.asarray(reverse_lengths)\n        self.batch_size = batch_size\n        self.rank = rank\n        self.world_size = world_size\n        self.weights = weights\n        self.bucket_size = max(batch_size, batch_size * bucket_size_multiplier)\n        self.seed = seed\n        self.epoch = 0\n\n    def set_epoch(self, epoch: int) -> None:\n        self.epoch = epoch\n\n    def __len__(self) -> int:\n        local_size = math.ceil(len(self.forward_lengths) / self.world_size)\n        return math.ceil(local_size / self.batch_size)\n\n    def __iter__(self) -> Iterator[list[tuple[int, bool]]]:\n        rng = np.random.default_rng(self.seed + self.epoch)\n        size = len(self.forward_lengths)\n        local_size = math.ceil(size / self.world_size)\n        total_size = local_size * self.world_size\n        if self.weights is None:\n            indices = rng.permutation(size)\n            if total_size > size:\n                indices = np.concatenate([indices, indices[: total_size - size]])\n        else:\n            indices = rng.choice(size, size=total_size, replace=True, p=self.weights)\n        local_indices = indices[self.rank : total_size : self.world_size]\n        orientations = rng.integers(0, 2, size=len(local_indices), dtype=np.int8).astype(bool)\n        keys = list(zip(local_indices.tolist(), orientations.tolist()))\n\n        ordered: list[tuple[int, bool]] = []\n        for start in range(0, len(keys), self.bucket_size):\n            bucket = keys[start : start + self.bucket_size]\n            bucket.sort(\n                key=lambda key: (\n                    self.reverse_lengths[key[0]]\n                    if key[1]\n                    else self.forward_lengths[key[0]]\n                )\n            )\n            ordered.extend(bucket)\n        batches = [\n            ordered[start : start + self.batch_size]\n            for start in range(0, len(ordered), self.batch_size)\n        ]\n        rng.shuffle(batches)\n        yield from batches\n\n\nclass FixedLengthBatchSampler(Sampler[list[tuple[int, bool]]]):\n    """Deterministic, padding-efficient batches for final validation."""\n\n    def __init__(\n        self,\n        cache: TokenCache,\n        pair_indices: Sequence[int],\n        batch_size: int,\n        both_orientations: bool = True,\n    ) -> None:\n        keys = [(int(index), False) for index in pair_indices]\n        if both_orientations:\n            keys.extend((int(index), True) for index in pair_indices)\n        forward_lengths = cache.forward_lengths\n        reverse_lengths = cache.reverse_lengths\n        keys.sort(\n            key=lambda key: reverse_lengths[key[0]] if key[1] else forward_lengths[key[0]]\n        )\n        self.batches = [\n            keys[start : start + batch_size]\n            for start in range(0, len(keys), batch_size)\n        ]\n\n    def __len__(self) -> int:\n        return len(self.batches)\n\n    def __iter__(self) -> Iterator[list[tuple[int, bool]]]:\n        yield from self.batches\n', 'src/validation_metrics.py': '"""Shared binary metrics for the frozen IID/hard/OOD validation protocol."""\n\nfrom __future__ import annotations\n\nfrom typing import Any\n\nimport numpy as np\nfrom sklearn.metrics import log_loss, precision_recall_curve, roc_auc_score\n\n\ndef _binary_arrays(\n    target: np.ndarray,\n    probability: np.ndarray,\n) -> tuple[np.ndarray, np.ndarray]:\n    target = np.asarray(target, dtype=np.int8).reshape(-1)\n    probability = np.asarray(probability, dtype=np.float64).reshape(-1)\n    if len(target) == 0 or len(target) != len(probability):\n        raise ValueError("target and probability must have the same non-zero length")\n    if not set(np.unique(target)) <= {0, 1}:\n        raise ValueError("target must contain only binary 0/1 values")\n    if np.unique(target).size != 2:\n        raise ValueError("binary validation metrics require both target classes")\n    if not np.isfinite(probability).all():\n        raise ValueError("probability must contain only finite values")\n    if (probability < 0).any() or (probability > 1).any():\n        raise ValueError("probability values must be in [0, 1]")\n    return target, probability\n\n\ndef recall_at_precision(\n    target: np.ndarray,\n    probability: np.ndarray,\n    *,\n    minimum_precision: float = 0.99,\n) -> tuple[float, float | None]:\n    """Return the maximum recall at a score threshold meeting precision floor."""\n    if not 0 < minimum_precision <= 1:\n        raise ValueError("minimum_precision must be in (0, 1]")\n    target, probability = _binary_arrays(target, probability)\n    precision, recall, thresholds = precision_recall_curve(target, probability)\n    # precision/recall include a final no-positive operating point without a\n    # corresponding threshold. Exclude it so an unavailable P99 returns R=0.\n    eligible = np.flatnonzero(precision[:-1] >= minimum_precision)\n    if eligible.size == 0:\n        return 0.0, None\n    eligible_recall = recall[eligible]\n    best_recall = float(eligible_recall.max())\n    # With equal recall prefer the lower threshold: it is the least restrictive\n    # operating point and remains deterministic when scores contain ties.\n    best_index = int(eligible[eligible_recall == best_recall][0])\n    return best_recall, float(thresholds[best_index])\n\n\ndef binary_probability_metrics(\n    target: np.ndarray,\n    probability: np.ndarray,\n    *,\n    minimum_precision: float = 0.99,\n) -> dict[str, Any]:\n    """Return protocol metrics derived from one split\'s probabilities."""\n    target, probability = _binary_arrays(target, probability)\n    recall, threshold = recall_at_precision(\n        target,\n        probability,\n        minimum_precision=minimum_precision,\n    )\n    clipped = np.clip(probability, 1e-15, 1 - 1e-15)\n    precision_label = f"{minimum_precision:.2f}".replace(".", "_")\n    return {\n        f"recall_at_precision_{precision_label}": recall,\n        f"threshold_at_precision_{precision_label}": threshold,\n        "roc_auc": float(roc_auc_score(target, probability)),\n        "log_loss": float(log_loss(target, clipped, labels=[0, 1])),\n    }\n'}
PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
for relative, content in EMBEDDED_SOURCES.items():
    destination = PROJECT_ROOT / relative
    destination.parent.mkdir(parents=True, exist_ok=True)
    destination.write_text(content, encoding="utf-8")
actual_source_hash = hashlib.sha256(
    json.dumps(
        EMBEDDED_SOURCES,
        ensure_ascii=False,
        sort_keys=True,
        separators=(",", ":"),
    ).encode("utf-8")
).hexdigest()
if actual_source_hash != EXPECTED_SOURCE_SHA256:
    raise RuntimeError("Embedded source fingerprint changed")
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--quiet",
        "--disable-pip-version-check",
        "--upgrade-strategy",
        "only-if-needed",
        "-r",
        str(PROJECT_ROOT / "requirements-cross-encoder.txt"),
    ],
    check=True,
)

## Prepare every human pair — no split and no filtering

In [ ]:
import numpy as np
import pandas as pd
sys.path.insert(0, str(PROJECT_ROOT))
from src.data_pipeline import serialize_product

preparation_started = time.perf_counter()
items = pd.read_parquet(
    items_path, columns=["id", "name", "attributes", "category"]
)
pairs = pd.read_parquet(matches_path, columns=["id1", "id2", "target"])
if len(items) != EXPECTED_DATA["items_rows"]:
    raise RuntimeError(f"Unexpected item count: {len(items)}")
if len(pairs) != EXPECTED_DATA["pairs_rows"]:
    raise RuntimeError(f"Unexpected pair count: {len(pairs)}")
if items["id"].duplicated().any() or items["id"].isna().any():
    raise RuntimeError("Item IDs must be unique and non-null")
if pairs[["id1", "id2", "target"]].isna().any().any():
    raise RuntimeError("Human pairs contain nulls")
if (pairs["id1"] == pairs["id2"]).any():
    raise RuntimeError("Human pairs contain self-pairs")
if not pairs["target"].isin([0.0, 1.0]).all():
    raise RuntimeError("Human labels must be binary")
known_ids = set(items["id"])
pair_ids = set(pairs["id1"]) | set(pairs["id2"])
if missing_ids := pair_ids - known_ids:
    raise RuntimeError(f"Missing product IDs: {len(missing_ids)}")
left = np.minimum(pairs["id1"].to_numpy(), pairs["id2"].to_numpy())
right = np.maximum(pairs["id1"].to_numpy(), pairs["id2"].to_numpy())
unordered = pd.DataFrame({"left": left, "right": right, "target": pairs["target"]})
grouped = unordered.groupby(["left", "right"], sort=False)["target"].agg(
    ["size", "nunique"]
)
duplicate_pairs = int((grouped["size"] > 1).sum())
contradictory_pairs = int((grouped["nunique"] > 1).sum())
if duplicate_pairs or contradictory_pairs:
    raise RuntimeError(
        f"Duplicate/conflicting pairs: {duplicate_pairs}/{contradictory_pairs}"
    )
categories = items.set_index("id")["category"]
if not pairs["id1"].map(categories).equals(pairs["id2"].map(categories)):
    raise RuntimeError("Human data contains cross-category pairs")

items = items.copy()
items["product_text"] = items.apply(
    serialize_product, axis=1, max_attribute_chars=6000
)
if not items["product_text"].str.startswith("Категория: ").all():
    raise RuntimeError("Serialization must start with Категория")
if not items["product_text"].str.contains("\nНазвание: ", regex=False).all():
    raise RuntimeError("Serialization must contain Название on line two")

PREPARED_DIR.mkdir(parents=True, exist_ok=True)
items[["id", "product_text", "category"]].to_parquet(
    PREPARED_DIR / "items.parquet", index=False, compression="zstd"
)
pairs.to_parquet(
    PREPARED_DIR / "train_pairs.parquet", index=False, compression="zstd"
)
preparation_report = {
    "items": len(items),
    "train_pairs": len(pairs),
    "positive_pairs": int((pairs["target"] == 1.0).sum()),
    "negative_pairs": int((pairs["target"] == 0.0).sum()),
    "duplicate_unordered_pairs": duplicate_pairs,
    "contradictory_pairs": contradictory_pairs,
    "validation_pairs": 0,
    "filtered_pairs": 0,
    "serialization": "Категория + Название + JSON key/value lines",
    "elapsed_seconds": time.perf_counter() - preparation_started,
}
(WORKING_ROOT / "full_human_data_report.json").write_text(
    json.dumps(preparation_report, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
print(json.dumps(preparation_report, ensure_ascii=False, indent=2))
print(items["product_text"].iloc[0][:2000])
del items, pairs, categories, unordered, grouped, known_ids, pair_ids

## 2×T4 DDP full fine-tuning

In [ ]:
train_command = [
    sys.executable,
    "-m",
    "torch.distributed.run",
    "--standalone",
    "--nproc_per_node=2",
    str(PROJECT_ROOT / "scripts/train_cross_encoder.py"),
    "--config",
    str(CONFIG_PATH),
    "--prepared-dir",
    str(PREPARED_DIR),
    "--output-dir",
    str(OUTPUT_DIR),
    "--token-cache-dir",
    str(TOKEN_CACHE_DIR),
]
training_environment = os.environ.copy()
training_environment.update({
    "OMP_NUM_THREADS": "2",
    "TOKENIZERS_PARALLELISM": "true",
    "NCCL_DEBUG": "WARN",
    "PYTHONUNBUFFERED": "1",
})
print("$", " ".join(train_command), flush=True)
wall_started = time.perf_counter()
with TRAIN_LOG.open("w", encoding="utf-8", buffering=1) as log_file:
    process = subprocess.Popen(
        train_command,
        cwd=PROJECT_ROOT,
        env=training_environment,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end="", flush=True)
        log_file.write(line)
    return_code = process.wait()
if return_code:
    raise subprocess.CalledProcessError(return_code, train_command)
training_wall_seconds = time.perf_counter() - wall_started

## Completion marker (local Kaggle output only)

In [ ]:
report_path = OUTPUT_DIR / "training_report.json"
if not report_path.is_file():
    raise RuntimeError("Training finished without training_report.json")
report = json.loads(report_path.read_text(encoding="utf-8"))
if report["original_training_examples"] != EXPECTED_DATA["pairs_rows"]:
    raise RuntimeError("Trainer did not receive every human pair")
if report["validation_splits"] or report["validation_examples"] != 0:
    raise RuntimeError("Final run unexpectedly evaluated validation data")
args = report["args"]
required = LOCKED_RECIPE
differences = {
    key: {"expected": value, "actual": args.get(key)}
    for key, value in required.items()
    if args.get(key) != value
}
if differences:
    raise RuntimeError(f"Locked final recipe changed: {differences}")
if report["world_size"] != EXPECTED_WORLD_SIZE:
    raise RuntimeError(
        f"Expected {EXPECTED_WORLD_SIZE} GPUs, got {report['world_size']}"
    )
effective_batch_size = (
    args["batch_size"]
    * report["world_size"]
    * args["gradient_accumulation"]
)
if effective_batch_size != EXPECTED_EFFECTIVE_BATCH_SIZE:
    raise RuntimeError(
        "Unexpected effective batch size: "
        f"{effective_batch_size} != {EXPECTED_EFFECTIVE_BATCH_SIZE}"
    )
if EXPECTED_AMP_DTYPE and report["amp_dtype"] != EXPECTED_AMP_DTYPE:
    raise RuntimeError(
        f"Expected {EXPECTED_AMP_DTYPE}, got {report['amp_dtype']}"
    )
if report["loss_hook"]["path"] is not None:
    raise RuntimeError("Final run must use the built-in BCE loss")
if not (
    report["training_loss_weight_min"]
    == report["training_loss_weight_median"]
    == report["training_loss_weight_max"]
    == 1.0
):
    raise RuntimeError("Final run unexpectedly applied sample weights")
completion = {
    "status": "complete",
    "completed_at_utc": datetime.now(timezone.utc).isoformat(
        timespec="seconds"
    ).replace("+00:00", "Z"),
    "dataset_ref": EXPECTED_DATASET_REF,
    "checkpoint_dataset_ref": EXPECTED_CHECKPOINT_DATASET_REF,
    "source_sha256": EXPECTED_SOURCE_SHA256,
    "training_wall_seconds": training_wall_seconds,
    "preparation_report": preparation_report,
    "training_report": report,
}
completion_path = WORKING_ROOT / "notebook_completed.json"
completion_path.write_text(
    json.dumps(completion, ensure_ascii=False, indent=2, default=str),
    encoding="utf-8",
)
print(json.dumps({
    "status": "complete",
    "model_dir": str(OUTPUT_DIR),
    "train_pairs": report["original_training_examples"],
    "epochs": args["epochs"],
    "training_hours": training_wall_seconds / 3600,
}, ensure_ascii=False, indent=2))